<p class="lead">Every project begins with the same unglamorous task: getting the data out of somebody else's system and into a table you can trust. This unit covers the four routes the laboratory uses — reading a file, calling a web interface, querying a relational database, and mapping identifiers between all of them — and treats each with the same question in mind: what exactly did I receive, and can I prove it later? We measure the file formats against each other instead of asserting their properties; we call a public API live with <code>httr2</code>, with the timeout, retry and caching discipline that a shared public service deserves; we build a small relational database from the course's frozen tables with <code>DBI</code> and <code>RSQLite</code>, and answer the same question twice, once in <code>dplyr</code> and once in SQL, and compare the two answers value by value.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata de cómo obtener los datos. Comparamos formatos midiéndolos: CSV, TSV comprimido, JSON (<code>jsonlite</code>), Parquet (<code>arrow</code>) y RDS, con sus tamaños y tiempos de lectura y escritura, y mostramos qué pierde un CSV (los tipos de dato) y qué conserva Parquet. Presentamos los formatos del dominio — SMILES y SDF para moléculas, FASTA para secuencias, PDB para estructuras — leyendo archivos reales del curso: el SDF congelado de los compuestos de ROCK1, un FASTA de familias de proteínas y la estructura del Trp-cage. Luego la web: anatomía de una petición HTTP, códigos de estado, paginación, límites de uso, reintentos con espera exponencial y almacenamiento en caché; <strong>esta es la única unidad del curso que hace llamadas reales a Internet</strong>, siempre dentro de <code>tryCatch()</code> con 30 s de espera máxima y con el archivo congelado del curso como respaldo. Seguimos con los sistemas de identificadores (ChEMBL ID, PubChem CID, UniProt, PDB, GEO, InChIKey) y por qué el mapeo entre ellos es la fuente de error más frecuente de un proyecto integrativo, midiendo en la tabla curada cuántas estructuras comparten el bloque de esqueleto del InChIKey. Terminamos con SQL en una hora sobre una base SQLite construida en el propio documento, y respondemos la misma pregunta con <code>dplyr</code> y con SQL para comparar ambos resultados.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Read and write the formats that circulate in the group — CSV/TSV, JSON, Parquet, RDS — and choose one for a given size and fidelity requirement, on measured evidence.</li>
<li>Recognise the domain formats you will meet (SMILES, SDF, FASTA, PDB/mmCIF) and state what each stores, having parsed a real example of each.</li>
<li>Query a public REST API with pagination, rate limiting, retries with exponential backoff and a disk cache, and always with a frozen fallback.</li>
<li>Name the identifier systems of the field (ChEMBL ID, PubChem CID, UniProt accession, PDB ID, GEO accession, InChIKey), and explain why mapping between them is error-prone.</li>
<li>Write SQL: SELECT, WHERE, ORDER BY, JOIN, GROUP BY, HAVING and subqueries; and say when a database beats a folder of CSV files.</li>
<li>Reproduce one answer in both <code>dplyr</code> and SQL and verify that they agree.</li>
<li>Respect terms of use, licences and <code>robots.txt</code>, and recognise when scraping is not acceptable.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running. <strong>This is the only unit of the course whose code touches the network.</strong> Every live call is wrapped in <code>tryCatch()</code> with a 30-second timeout and falls back to the frozen copy in <code>../data/</code>, and each such chunk prints which path it took — so the document is deterministic for a reader without a connection, and honest about it. Beyond the Unit 1 stack this unit uses <code>jsonlite</code>, <code>arrow</code> (Parquet), <code>httr2</code> (HTTP), <code>DBI</code> with <code>RSQLite</code> (SQL) and <code>seqinr</code> (FASTA); in a fresh Colab R runtime install them once with <code>install.packages(c("arrow","httr2","RSQLite","seqinr","data.table"))</code>. Temporary files are written to a scratch directory removed in the last chunk. This is the <strong>R edition</strong>; its <a href="../../Python/U3_Getting_Data/index.html">Python counterpart</a> performs the same measurements with pandas, <code>requests</code>, <code>h5py</code> and <code>sqlite3</code>, and every number below was produced by the R code shown here.</p></div>



In [ ]:
## Setup. In a fresh Colab R runtime, install arrow, httr2, RSQLite and seqinr first.
suppressPackageStartupMessages({
  library(data.table); library(dplyr); library(ggplot2)
  library(jsonlite); library(arrow); library(httr2); library(DBI); library(RSQLite)
  library(seqinr); library(digest)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9.5) +
          theme(plot.title = element_text(face = "bold", size = 10.5),
                panel.grid.minor = element_blank()))

REPO <- "DanielBustosG/DanielBustosG.github.io"
DATA_URL <- sprintf("https://raw.githubusercontent.com/%s/main/DataScience/data/", REPO)
data_path <- function(name) {
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dir.create("data", showWarnings = FALSE); dest <- file.path("data", name)
  if (!file.exists(dest)) download.file(paste0(DATA_URL, name), dest, quiet = TRUE)
  dest
}
load_data <- function(name, ...) data.table::fread(data_path(name), ...)

SCRATCH <- normalizePath(file.path(tempdir(), "u3_scratch"), mustWork = FALSE)
CACHE <- file.path(SCRATCH, "api_cache")
dir.create(CACHE, recursive = TRUE, showWarnings = FALSE)
TIMEOUT <- 30                       # seconds; every network call in this document uses it
CHEMBL <- "https://www.ebi.ac.uk/chembl/api/data"

raw <- load_data("rock1_chembl_raw.csv")
for (p in c("data.table", "jsonlite", "arrow", "httr2", "RSQLite", "seqinr"))
  cat(sprintf("%-11s %s\n", p, as.character(packageVersion(p))))
cat("frozen ROCK1 activity table:", dim(raw), "(rows, columns)\n")
cat("scratch directory:", basename(SCRATCH), "\n")



## 1. The data landscape of the laboratory

Before any format or protocol, a map. A project in this group draws on five kinds of public resource, each with its own content, curation policy, release cycle and — most consequentially — its own identifiers.

- **ChEMBL** (EMBL-EBI) holds bioactivity measurements extracted and standardised from the medicinal-chemistry literature: compounds, targets, assays, documents and activities, with roughly twenty million activity records in recent releases<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>. It is curated by humans, versioned by release, and licensed CC BY-SA. Its identifiers look like `CHEMBL3231` (a target) or `CHEMBL1201305` (a molecule), and its curation policy — what counts as the same assay, which values are flagged — is documented in detail<sup class="c"><a href="#r4">[4]</a></sup>.
- **PubChem** (NCBI) is larger and less curated: it aggregates depositor-submitted substances and computed compounds, with bioassay results attached<sup class="c"><a href="#r5">[5]</a></sup><sup class="c"><a href="#r6">[6]</a></sup>. Its identifiers are the **CID** (a compound) and **SID** (a deposited substance), and the distinction between them is a classic source of error.
- **BindingDB** curates binding affinities with an emphasis on measured \( K_i \) and \( K_d \) values, and cross-references ChEMBL and PubChem<sup class="c"><a href="#r7">[7]</a></sup>.
- **UniProt** is the reference for proteins: one accession (such as `Q13464` for human ROCK1) per sequence entry, with annotation, cross-references and isoform handling<sup class="c"><a href="#r8">[8]</a></sup>. **RCSB PDB** holds the experimental three-dimensional structures, named by a four-character code such as `2ESM`, and now also computed models<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r10">[10]</a></sup>.
- **GEO** (NCBI) archives functional-genomics experiments: a series (`GSE16134`), its samples (`GSM…`) and the platform (`GPL570`) on which they were measured<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r12">[12]</a></sup><sup class="c"><a href="#r13">[13]</a></sup>.

<div class="box bridge"><h4>Domain context — the file formats you will be handed</h4>
<p>Four formats carry the domain objects of this laboratory, and you will meet all four in your first month.</p>
<ul>
<li><strong>SMILES</strong> is a one-line text encoding of a molecular graph: <code>CC(=O)Oc1ccccc1C(=O)O</code> is aspirin. Atoms are element symbols, branches are parentheses, rings are opened and closed by matching digits, lower case means aromatic<sup class="c"><a href="#r14">[14]</a></sup>. It fits in a CSV column, which is why every table in this course stores structures as SMILES.</li>
<li><strong>SDF</strong> (structure-data file, from the MDL CTfile family<sup class="c"><a href="#r15">[15]</a></sup>) is the multi-record text format that chemistry software exchanges: per molecule, a title line, two header lines, a counts line, one line per atom with coordinates, one line per bond, then arbitrary named data fields terminated by <code>$$$$</code>. Use it when coordinates or per-molecule annotations must travel with the structures.</li>
<li><strong>FASTA</strong> is the sequence format: a header line beginning with <code>&gt;</code> followed by the residue letters of a protein or nucleic acid<sup class="c"><a href="#r16">[16]</a></sup>. One file holds many sequences; nothing in it is typed, which is why sequence identifiers in FASTA headers are a recurring data-cleaning task.</li>
<li><strong>PDB</strong> and <strong>mmCIF</strong> hold macromolecular structures — atoms with coordinates, plus metadata about how they were determined. The legacy PDB format is a fixed-column text format whose 80-character lines cannot express large structures; mmCIF is the dictionary-based successor and is now the archival format<sup class="c"><a href="#r10">[10]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>.</li>
</ul>
<p>This unit handles them as <em>formats</em> — how to recognise, read and write them. The chemistry and structural biology are the subject of the sister courses.</p></div>



In [ ]:
## Conceptual figure: what each resource holds, and the identifier that leaves it.
items <- data.frame(
  x = 1:5,
  name = c("ChEMBL", "PubChem", "BindingDB", "UniProt /\nRCSB PDB", "GEO"),
  content = c("curated bioactivities\nfrom the literature",
              "deposited substances,\ncomputed compounds, assays",
              "curated binding\naffinities (Ki, Kd)",
              "protein sequences,\nannotation, 3-D structures",
              "expression series,\nsamples, platforms"),
  ids = c("CHEMBL3231\nCHEMBL1201305", "CID 2244\nSID 24892019", "BDBM id\n+ cross-refs",
          "Q13464\n2ESM", "GSE16134\nGPL570"),
  fill = c("#dce8f5", "#f7e7d6", "#f6e6d8", "#dcefdc", "#ece7f6"), stringsAsFactors = FALSE)

p <- ggplot(items) +
  geom_tile(aes(x, 1.18, fill = I(fill)), width = 0.9, height = 1.25, colour = "grey35",
            linewidth = 0.4) +
  geom_text(aes(x, 1.62, label = name), fontface = "bold", size = 2.9) +
  geom_text(aes(x, 1.15, label = content), size = 2.3) +
  geom_text(aes(x, 0.72, label = ids), size = 2.3, family = "mono", colour = PAL[["blue"]]) +
  geom_segment(aes(x = x, xend = x, y = 0.5, yend = 0.2), colour = "grey55",
               arrow = arrow(length = unit(0.07, "cm"), type = "closed")) +
  annotate("tile", x = 3, y = -0.1, width = 5.1, height = 0.5, fill = "#f7f7f7",
           colour = PAL[["orange"]], linewidth = 0.5) +
  annotate("text", x = 3, y = -0.1, size = 2.5, colour = PAL[["orange"]],
           label = "your project table: one row per compound, one column per identifier\nand every join a chance to be wrong") +
  scale_x_continuous(limits = c(0.4, 5.6)) + scale_y_continuous(limits = c(-0.45, 1.9)) +
  labs(title = "The public data landscape, and the identifiers it emits") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)



<p class="caption"><b>Figure 1.</b> The five families of public resource this laboratory draws on, what each contains, and the identifiers it emits. Every arrow into the bottom box is an identifier mapping, and Section 5 shows why each one is a chance to be silently wrong. Note that the same molecule may appear as a ChEMBL id, a PubChem CID and a BindingDB entry — three names for one chemical object — while the same protein appears as a UniProt accession and several PDB codes, one per crystal structure.</p>

## 2. Formats, measured

Formats differ along four axes: **text or binary**, **with or without a schema** (does the file record the type of each column?), **row-oriented or column-oriented**, and **compressed or not**. Rather than assert the consequences, we measure them on the real 5,089-row ChEMBL table. The Python edition compares CSV, gzipped TSV, JSON, Parquet and HDF5; here HDF5 is replaced by **RDS**, R's own binary serialisation format, which plays the same role in R workflows and is the one you will actually meet in a colleague's project folder. HDF5 itself is readable in R through the Bioconductor package `rhdf5`, which is not installed here; the trade-off it represents — a typed binary container for large numeric arrays — is the same one RDS illustrates.



In [ ]:
## Write the same table in five formats and measure size and write time.
numeric_cols <- names(raw)[vapply(raw, is.numeric, logical(1))]

writers <- list(
  "CSV" = function(p) fwrite(raw, p),
  "TSV (gzip)" = function(p) fwrite(raw, p, sep = "\t", compress = "gzip"),
  "JSON (records)" = function(p) write_json(raw, p, dataframe = "rows"),
  "Parquet" = function(p) write_parquet(raw, p),
  "RDS (compressed)" = function(p) saveRDS(raw, p, compress = "gzip"))
exts <- c(CSV = "csv", `TSV (gzip)` = "tsv.gz", `JSON (records)` = "json",
          Parquet = "parquet", `RDS (compressed)` = "rds")

rows <- list()
for (nm in names(writers)) {
  path <- file.path(SCRATCH, paste0("rock1.", exts[[nm]]))
  t0 <- proc.time()[["elapsed"]]; writers[[nm]](path); dt <- proc.time()[["elapsed"]] - t0
  rows[[nm]] <- data.frame(format = nm, MB = round(file.size(path) / 1e6, 3),
                           write_s = round(dt, 3),
                           typed = if (nm %in% c("Parquet", "RDS (compressed)")) "yes" else "no",
                           columns = "all 25")
}
sizes <- do.call(rbind, rows)
print(knitr::kable(sizes, row.names = FALSE, format = "simple"))
cat(sprintf("\nParquet is %.1f times smaller than CSV for the same 25 columns; RDS is %.1f times smaller.\n",
            sizes$MB[sizes$format == "CSV"] / sizes$MB[sizes$format == "Parquet"],
            sizes$MB[sizes$format == "CSV"] / sizes$MB[sizes$format == "RDS (compressed)"]))
cat("numeric columns in the table:", length(numeric_cols), "of", ncol(raw), "\n")



<p class="caption"><b>Table (Section 2).</b> The same 5,089 × 25 table written five ways: file size, write time, whether the format records column types, and which columns it can hold. Unlike the Python edition's HDF5 row, every format here holds all twenty-five columns, because RDS serialises an arbitrary R object rather than a numeric array — which is simultaneously its convenience and its weakness, as the next paragraphs explain.</p>



In [ ]:
## Figure: size and read time per format, measured (best of three reads).
readers <- list("CSV" = function(p) fread(p),
                "TSV (gzip)" = function(p) fread(cmd = paste("gzip -dc", shQuote(p))),
                "JSON (records)" = function(p) fromJSON(p),
                "Parquet" = function(p) read_parquet(p),
                "RDS (compressed)" = function(p) readRDS(p))
best_of <- function(f, repeats = 3) {
  b <- Inf
  for (i in seq_len(repeats)) { t0 <- proc.time()[["elapsed"]]; f(); b <- min(b, proc.time()[["elapsed"]] - t0) }
  b
}
read_times <- vapply(names(readers), function(nm)
  best_of(function() readers[[nm]](file.path(SCRATCH, paste0("rock1.", exts[[nm]])))), numeric(1))

d <- data.frame(format = factor(names(readers), levels = rev(names(readers))),
                MB = sizes$MB[match(names(readers), sizes$format)],
                ms = as.numeric(read_times) * 1000)
library(patchwork)
p1 <- ggplot(d, aes(MB, format)) + geom_col(fill = PAL[["blue"]], width = 0.65) +
  geom_text(aes(label = sprintf(" %.2f", MB)), hjust = 0, size = 2.6) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.18))) +
  labs(x = "file size (MB)", y = NULL, title = "Size of the same 5,089-row table")
p2 <- ggplot(d, aes(ms, format)) + geom_col(fill = PAL[["orange"]], width = 0.65) +
  geom_text(aes(label = sprintf(" %.0f ms", ms)), hjust = 0, size = 2.6) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.22))) +
  labs(x = "read time (ms, best of 3)", y = NULL, title = "Time to read it back")
print(p1 | p2)
cat(paste(sprintf("%s: %.0f ms", names(read_times), read_times * 1000), collapse = " | "), "\n")



<p class="caption"><b>Figure 2.</b> File size (left) and read time (right) for the same table in five formats, measured on the machine that built this page. Parquet is among the smallest and the fastest, because it stores each column separately in a typed, compressed, binary layout; JSON is the largest because it repeats every column name in every record; gzipped TSV is small but pays decompression on every read; RDS is compact but must reconstruct a complete R object. CSV read by <code>fread</code> is competitive on time — <code>fread</code> is a multi-threaded C parser, so the gap between text and binary is narrower in R than the Python edition reports for <code>pandas.read_csv</code> — and wins outright on universality. The numbers printed under the figure are the measured read times.</p>

The recommendation that follows from the measurement, and that this group applies: **CSV for anything a human or a stranger's tool must read** (raw downloads, small result tables, supplementary material), **Parquet for intermediate tables inside a pipeline** and for anything another language must also read<sup class="c"><a href="#r19">[19]</a></sup>, **HDF5 or a compressed binary array format for large numeric matrices** such as trajectories and fingerprint matrices<sup class="c"><a href="#r18">[18]</a></sup>, and **JSON for nested metadata** — a manifest, an API response — where the structure is a tree rather than a rectangle<sup class="c"><a href="#r20">[20]</a></sup>. RDS has one legitimate niche and one trap: it is the right way to cache a *fitted model* or any R object that is not a table, and it is the wrong way to exchange a table with anyone, because only R can read it and the file silently depends on the R serialisation version and on whatever packages defined the classes inside it.

### 2.1 What a CSV loses

A CSV file stores characters. The *types* — this column is an integer, that one is a date, this identifier must keep its leading zero — exist only in the reader's guess. The next chunk exhibits the failure on constructed data of exactly the kinds that occur in this field.



In [ ]:
## The CSV type-loss experiment: five columns that do not survive a round trip.
original <- data.frame(
  plate_well   = c("0012", "0345", "1000"),            # an identifier with meaningful leading zeros
  n_replicates = c(3L, NA_integer_, 5L),               # an integer column with a missing value
  measured_on  = as.Date(c("2026-01-15", "2026-02-02", "2026-03-30")),
  is_active    = c(TRUE, FALSE, TRUE),
  comment      = c("ok", "NA", "n.d."),                # literal text that looks like missingness
  stringsAsFactors = FALSE)
fwrite(original, file.path(SCRATCH, "types.csv"))
roundtrip <- as.data.frame(fread(file.path(SCRATCH, "types.csv")))

cat("class before -> after:\n")
for (col in names(original))
  cat(sprintf("  %-13s %-10s -> %s\n", col, class(original[[col]])[1], class(roundtrip[[col]])[1]))
cat("\nvalues that changed meaning:\n")
cat("  plate_well  :", original$plate_well, "->", roundtrip$plate_well, "\n")
cat("  n_replicates:", format(original$n_replicates), "->", format(roundtrip$n_replicates), "\n")
cat("  comment     :", original$comment, "->", ifelse(is.na(roundtrip$comment), "<NA>",
                                                      roundtrip$comment), "\n")

## Parquet keeps the schema: the round trip is exact.
write_parquet(original, file.path(SCRATCH, "types.parquet"))
pq <- as.data.frame(read_parquet(file.path(SCRATCH, "types.parquet")))
cat("\nParquet round trip preserves every class:",
    all(vapply(names(original), function(c) identical(class(original[[c]]), class(pq[[c]])),
               logical(1))), "\n")
cat("and every value:", isTRUE(all.equal(original, pq, check.attributes = FALSE)), "\n")



Read the losses one by one, because each has corrupted a real dataset. The identifier `0012` came back as the integer 12, so a join against the plate map now fails for every well below 1000. The integer column with a missing value survived as an integer in R (unlike pandas, where it would become a float) — this is one of the few places where R's reader is better behaved than Python's, because R has a genuine `NA` for every type. The date came back as a `Date` here only because `fread` recognises the ISO format; written in any other format it returns text. The logical column survived. And the literal comment `"NA"` was read as *missing*, silently converting a curator's note into an absence — the one failure that is identical in both languages and the most damaging, because nothing downstream can detect it. The Parquet round trip, by contrast, is exact in both classes and values, because the format carries a schema.

<div class="box warn"><h4>The three defences when you must use CSV</h4>
<p>Pass explicit types: <code>fread(path, colClasses = c(plate_well = "character"))</code>, or <code>readr::read_csv(path, col_types = cols(plate_well = col_character()))</code>. Name the date columns, because <code>fread</code> guesses only ISO-8601 formats. And set <code>na.strings = ""</code> when a column may legitimately contain the strings <code>NA</code>, <code>None</code> or <code>null</code> as data — <code>fread</code>'s default <code>na.strings = c("", "NA")</code> is exactly what destroyed the comment column above. A reader without these arguments is a guess, and the guess differs between <code>fread</code>, <code>read.csv</code> and <code>read_csv</code>.</p></div>



In [ ]:
## Column pruning: the practical advantage of a columnar format on a wide table.
wide <- rbindlist(rep(list(raw), 20))                       # about 101,780 rows x 25 columns
write_parquet(wide, file.path(SCRATCH, "wide.parquet"))
fwrite(wide, file.path(SCRATCH, "wide.csv"))
cat("wide table:", dim(wide), sprintf("| parquet %.1f MB | csv %.1f MB\n",
    file.size(file.path(SCRATCH, "wide.parquet")) / 1e6,
    file.size(file.path(SCRATCH, "wide.csv")) / 1e6))

two_cols <- c("molecule_chembl_id", "standard_value")
timings <- c(
  "parquet, all 25 columns" = best_of(function() read_parquet(file.path(SCRATCH, "wide.parquet"))),
  "parquet, 2 columns" = best_of(function() read_parquet(file.path(SCRATCH, "wide.parquet"),
                                                         col_select = all_of(two_cols))),
  "csv, all 25 columns" = best_of(function() fread(file.path(SCRATCH, "wide.csv"))),
  "csv, 2 columns" = best_of(function() fread(file.path(SCRATCH, "wide.csv"), select = two_cols)))
for (k in names(timings)) cat(sprintf("  %-26s %7.0f ms\n", k, timings[[k]] * 1000))
cat(sprintf("\nreading 2 of 25 columns is %.1f times faster in Parquet, and %.1f times faster in CSV\n",
            timings[["parquet, all 25 columns"]] / timings[["parquet, 2 columns"]],
            timings[["csv, all 25 columns"]] / timings[["csv, 2 columns"]]))



Read that output in two directions, because the R result is more interesting than the Python edition's. **Down the columns**: Parquet is an order of magnitude faster than CSV in absolute terms, whether you ask for two columns or twenty-five, and the file it reads is a hundred times smaller — that is the columnar, compressed, dictionary-encoded layout doing its work. **Across the two rows of each format**: selecting two of twenty-five columns speeds up *both* readers by a similar factor, and in this measurement `fread`'s `select =` gains proportionally slightly more than Parquet's `col_select`. That is not a contradiction of the columnar argument; it says where each reader's time goes. Parquet already spends almost nothing, so there is less to save. The dominant cost of a full `fread` is not finding the separators — that scan is multi-threaded C — but allocating and type-converting twenty-five columns, several of which are long SMILES and assay-description strings; `select =` skips that construction while still scanning the bytes. The practical conclusion is unchanged, and stronger: read the columns you need, and prefer a columnar format for an intermediate table. On a table of a hundred thousand rows the gap is already visible; on the ten-million-row output of a virtual-screening campaign (Unit 19) it is the difference between an interactive analysis and a coffee break.

<div class="exercise"><h4>Exercise 1</h4>
<p>Write <code>wide</code> to Parquet three times, with <code>compression = "snappy"</code> (the default), <code>"gzip"</code> and <code>"uncompressed"</code>, and compare file size against write and read time. Then convert the three low-cardinality text columns to factors before writing and repeat: by how much does the Parquet file shrink, and why does dictionary encoding help a columnar format so much more than it helps a CSV? Finally, write the same table with <code>saveRDS(..., compress = "xz")</code> and explain, from the measured write time, why <code>xz</code> is the wrong default for a file you will regenerate every day.</p></div>



In [ ]:
## How the choice scales: size of each format as the table grows.
scaling <- do.call(rbind, lapply(c(1, 2, 5, 10, 20), function(factor) {
  big <- rbindlist(rep(list(raw), factor))
  p_csv <- file.path(SCRATCH, "s.csv"); p_pq <- file.path(SCRATCH, "s.parquet")
  fwrite(big, p_csv); write_parquet(big, p_pq)
  out <- data.frame(rows = nrow(big), csv_MB = file.size(p_csv) / 1e6,
                    parquet_MB = file.size(p_pq) / 1e6)
  unlink(c(p_csv, p_pq)); out
}))
scaling$ratio <- round(scaling$csv_MB / scaling$parquet_MB, 1)
print(knitr::kable(scaling, row.names = FALSE, digits = 2, format = "simple"))

long <- rbind(data.frame(rows = scaling$rows, MB = scaling$csv_MB, format = "CSV"),
              data.frame(rows = scaling$rows, MB = scaling$parquet_MB, format = "Parquet"))
p <- ggplot(long, aes(rows, MB, colour = format, shape = format)) +
  geom_line(linewidth = 0.7) + geom_point(size = 2) +
  geom_text(data = scaling, aes(x = rows, y = parquet_MB, label = paste0("x", ratio)),
            inherit.aes = FALSE, vjust = 1.9, size = 2.5, colour = PAL[["blue"]]) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")])) +
  labs(x = "rows in the table", y = "file size (MB)", colour = NULL, shape = NULL,
       title = "Both grow linearly; the constant is what differs") +
  theme(legend.position = "bottom")
print(p)



<p class="caption"><b>Figure 3.</b> File size against row count for CSV and Parquet, from the 5,089-row table up to twenty copies of it. Both are linear — no format changes the asymptotic behaviour — but the ratio annotated at each point is roughly constant and large, because the repeated SMILES strings and assay descriptions compress and dictionary-encode extremely well in a columnar layout. When someone asks "will this scale?", this is the shape of answer to give: the same complexity class, a different constant, and the constant is what you will feel.</p>

## 3. The domain formats in practice

The formats of Section 2 hold tables. The four domain formats hold objects: molecules, sequences, structures. The next chunks parse a real example of each from the course's own frozen files, so that the first time you are handed one of them, nothing about it is unfamiliar.

One substitution must be declared here. **R has no RDKit binding in this environment**, so SMILES strings are not turned into molecule objects, and nothing in this R edition computes a structure from a string. Structures therefore enter this course as data: the SMILES and InChIKey columns of the frozen tables, and `rock1_curated.sdf.gz`, a frozen SDF of the curated compounds generated with RDKit and documented in <a href="../../data/manifest_fingerprints.json">manifest_fingerprints.json</a>. A reader who installs `ChemmineR`<sup class="c"><a href="#r36">[36]</a></sup> can recompute everything from that SDF; this unit reads it as a text format, which is the skill it is teaching.



In [ ]:
## (a) SMILES as data, and (b) SDF parsed by hand from the frozen file.
smiles_sample <- raw[nzchar(canonical_smiles), canonical_smiles][1:3]
cat("(a) SMILES: three strings straight out of the frozen table\n")
for (s in smiles_sample) cat("    ", substr(s, 1, 68), if (nchar(s) > 68) "..." else "", "\n")
cat("    lengths:", nchar(smiles_sample), "characters; ring-closure digits in the first:",
    length(gregexpr("[0-9]", smiles_sample[1])[[1]]), "\n")

con <- gzfile(data_path("rock1_curated.sdf.gz"), "rt")
lines <- readLines(con, n = 400); close(con)
end <- which(lines == "$$$$")[1]
rec <- lines[1:(end - 1)]
counts <- rec[4]
n_atoms <- as.integer(substr(counts, 1, 3)); n_bonds <- as.integer(substr(counts, 4, 6))
atom_block <- rec[5:(4 + n_atoms)]
atoms <- data.frame(
  x = as.numeric(substr(atom_block, 1, 10)), y = as.numeric(substr(atom_block, 11, 20)),
  z = as.numeric(substr(atom_block, 21, 30)), element = trimws(substr(atom_block, 31, 34)))
field_lines <- grep("^> ", rec, value = TRUE)
cat("\n(b) SDF: first record of rock1_curated.sdf.gz\n")
cat("    title line (the InChIKey):", rec[1], "\n")
cat("    counts line:", trimws(counts), "-> atoms", n_atoms, "bonds", n_bonds, "\n")
cat("    element composition:", paste(names(table(atoms$element)), table(atoms$element),
                                      sep = "=", collapse = " "), "\n")
cat("    data fields:", paste(gsub("[<>]|^> ", "", field_lines), collapse = ", "), "\n")
cat("    records in the file are terminated by $$$$; this one ended at line", end, "\n")



Two parsing strategies and two lessons. A **SMILES** is a string: it fits in a CSV column, it is compared with `==`, and it means nothing to R without a chemistry toolkit — which is why the curated table ships the InChIKey as well, so that structures can be matched without parsing them. An **SDF** is a *record-oriented fixed-column text format*: the counts line declares how many atoms and bonds follow, the atom block uses fixed character positions for the coordinates (`substr(line, 1, 10)` is the x coordinate, not "the first field"), and the record ends with `$$$$`. Our fifteen-line parser handles the first record correctly; a production parser must additionally handle charge blocks, property blocks and V3000 files, which is exactly why `ChemmineR` exists.



In [ ]:
## (c) FASTA with seqinr, and (d) PDB parsed by column position.
fa <- read.fasta(data_path("protein_families.fasta"), seqtype = "AA", as.string = TRUE,
                 forceDNAtolower = FALSE)
headers <- vapply(fa, function(x) attr(x, "Annot"), character(1))
lens <- vapply(fa, function(x) nchar(x[[1]]), integer(1))
first_id <- strsplit(sub("^>", "", headers[1]), "\\|")[[1]]
cat("(c) FASTA:", length(fa), "records |", "residue counts: min", min(lens), "median",
    median(lens), "max", max(lens), "\n")
cat("    first header parsed on '|':", paste(first_id[1:2], collapse = " / "), "\n")
cat("    first 60 residues:", substr(fa[[1]][1], 1, 60), "\n")
aa <- table(strsplit(paste(unlist(fa), collapse = ""), "")[[1]])
cat("    three most frequent residues overall:",
    paste(names(sort(aa, decreasing = TRUE))[1:3],
          round(100 * sort(aa, decreasing = TRUE)[1:3] / sum(aa), 1), sep = "=", collapse = "  "),
    "%\n")

pdb_lines <- grep("^ATOM", readLines(data_path("md_trpcage_protein.pdb"), warn = FALSE),
                  value = TRUE)
atoms_pdb <- data.frame(
  serial  = as.integer(substr(pdb_lines, 7, 11)),
  name    = trimws(substr(pdb_lines, 13, 16)),
  resname = trimws(substr(pdb_lines, 18, 20)),
  chain   = substr(pdb_lines, 22, 22),
  resseq  = as.integer(substr(pdb_lines, 23, 26)),
  x = as.numeric(substr(pdb_lines, 31, 38)), y = as.numeric(substr(pdb_lines, 39, 46)),
  z = as.numeric(substr(pdb_lines, 47, 54)),
  bfactor = as.numeric(substr(pdb_lines, 61, 66)))
cat("\n(d) PDB:", nrow(atoms_pdb), "ATOM records |", length(unique(atoms_pdb$resseq)),
    "residues | chains:", paste(unique(atoms_pdb$chain), collapse = ","), "\n")
print(knitr::kable(head(atoms_pdb[, c("serial", "name", "resname", "resseq", "x", "y", "z")], 4),
                   row.names = FALSE, format = "simple"))
cat("splitting the same line on whitespace gives",
    length(strsplit(trimws(pdb_lines[1]), " +")[[1]]), "fields - correct here, and wrong",
    "as soon as a coordinate touches its neighbour.\n")



**FASTA** needs no special machinery — the format is a header line and residue lines — but `seqinr`<sup class="c"><a href="#r35">[35]</a></sup> gives you the records as a named list with the annotation preserved, which saves writing the parser everyone writes once. Note that the only hard part is the header convention: `read.fasta` returns the text after `>` verbatim, and extracting an accession from it is a string operation whose correctness depends on the source. **PDB** is a **fixed-column** format: the x coordinate is characters 31–38 of the line, not "the sixth field", so splitting on whitespace is wrong and breaks the moment a field touches its neighbour — the last printed line shows that it happens to give the right number of fields on *this* line, which is precisely how the bug survives testing. That fragility is exactly why mmCIF replaced PDB as the archival format<sup class="c"><a href="#r10">[10]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>; in R, `bio3d` reads both.

## 4. Web APIs: asking a server politely

A **REST API** exposes resources at URLs and returns representations of them, usually JSON<sup class="c"><a href="#r21">[21]</a></sup><sup class="c"><a href="#r20">[20]</a></sup>. ChEMBL's interface is documented as a research tool in its own right<sup class="c"><a href="#r32">[32]</a></sup>, and its deposition pipeline explains why the same assay can appear with different provenance in successive releases<sup class="c"><a href="#r33">[33]</a></sup>. The anatomy of a request is: a method (`GET` to read), a base URL, a path naming the resource, a query string of parameters, and headers. The response carries a status code — `200` success, `3xx` redirect, `400` your request was malformed, `404` no such resource, `429` you are being rate-limited, `5xx` the server failed — plus the body.

Four disciplines separate a well-behaved client from a nuisance, and public services such as the EBI and NCBI depend on users observing them.

1. **Timeouts.** Every call passes an explicit timeout; without one, a stalled connection hangs your document forever. In this document it is 30 seconds, set with `req_timeout()`.
2. **Pagination.** Large result sets are returned in pages. ChEMBL's API reports a `page_meta` block with the total count and the path of the next page; you loop until there is no next page, and you cap the loop.
3. **Retries with exponential backoff.** A transient `5xx` or `429` should be retried after a wait that doubles each time (1 s, 2 s, 4 s, …), with a small random jitter so that many clients do not retry in lockstep. A `404` should never be retried — the resource is simply not there. `httr2` has `req_retry()` for this, and we also implement the schedule by hand so that the policy is visible.
4. **Caching.** A response that will not change should be written to disk and re-used. This is politeness and speed at once: the second run of your analysis should not touch the network at all.



In [ ]:
## The backoff schedule, and the decision rule for which statuses deserve a retry.
RETRY_ON <- c(429, 500, 502, 503, 504)
sched <- data.frame(attempt = 1:6, delay = 2^(0:5))
sched$lo <- sched$delay * 0.8; sched$hi <- sched$delay * 1.2

p <- ggplot(sched, aes(attempt, delay)) +
  geom_ribbon(aes(ymin = lo, ymax = hi), fill = PAL[["blue"]], alpha = 0.22) +
  geom_step(colour = PAL[["blue"]], linewidth = 0.7, direction = "mid") +
  geom_point(size = 1.8, colour = PAL[["blue"]]) +
  geom_text(aes(label = paste0(delay, "s")), vjust = -0.9, size = 2.6) +
  scale_y_continuous(trans = "log2") +
  labs(x = "attempt number", y = "wait before retrying (s)",
       title = sprintf("Exponential backoff: cumulative wait of %.0f s over 6 attempts",
                       sum(sched$delay)),
       subtitle = "shaded band: +/- 20% jitter, which decorrelates many clients")
print(p)
cat("retry on:", paste(RETRY_ON, collapse = ", "),
    "| never retry: 400, 401, 403, 404 (the request itself is wrong)\n")
cat(sprintf("cumulative wait if all six attempts fail: %.0f s\n", sum(sched$delay)))



<p class="caption"><b>Figure 4.</b> The exponential-backoff schedule used by the client below, on a logarithmic axis, with the jitter band that prevents synchronised retries across many clients. Six attempts span a cumulative wait of about a minute — enough to ride out a brief server hiccup, short enough not to hide a real outage. The printed line states the rule that matters as much as the schedule: statuses that mean "your request is wrong" are never retried, because retrying them is pure load on a shared service.</p>



In [ ]:
## A polite httr2 client: timeout, retries with backoff and jitter, and a disk cache.
api_get <- function(url, params = list(), attempts = 3L, timeout = TIMEOUT, cache = CACHE) {
  ## GET with retries, backoff and caching. Returns list(payload, provenance); never stops.
  key <- paste0(digest(list(url, params), algo = "md5"), ".json")
  cached <- file.path(cache, key)
  if (file.exists(cached))
    return(list(payload = fromJSON(cached, simplifyVector = FALSE), provenance = "cache"))
  last <- ""
  for (attempt in seq_len(attempts)) {
    res <- tryCatch({
      req <- request(url) |> req_timeout(timeout) |>
        req_headers(Accept = "application/json") |>
        req_user_agent("LBQC-data-science-course (teaching material; contact the course author)") |>
        req_error(is_error = function(resp) FALSE)
      if (length(params)) req <- req |> req_url_query(!!!params)
      resp <- req_perform(req)
      list(status = resp_status(resp), body = resp_body_string(resp))
    }, error = function(e) list(status = -1L, body = paste0(class(e)[1], ": ",
                                                            substr(conditionMessage(e), 1, 60))))
    if (identical(res$status, 200L)) {
      writeLines(res$body, cached)
      return(list(payload = fromJSON(res$body, simplifyVector = FALSE),
                  provenance = sprintf("network (attempt %d)", attempt)))
    }
    if (res$status > 0 && !(res$status %in% RETRY_ON))
      return(list(payload = NULL, provenance = sprintf("HTTP %d (not retried)", res$status)))
    last <- if (res$status > 0) paste("HTTP", res$status) else res$body
    if (attempt < attempts) Sys.sleep(min(2^(attempt - 1), 4))
  }
  list(payload = NULL, provenance = sprintf("gave up after %d attempts (%s)", attempts, last))
}

got <- api_get(paste0(CHEMBL, "/activity.json"),
               list(target_chembl_id = "CHEMBL3231", standard_type = "IC50", limit = 5))
cat("provenance:", got$provenance, "\n")
if (!is.null(got$payload)) {
  pm <- got$payload$page_meta
  cat("top-level keys :", paste(names(got$payload), collapse = ", "), "\n")
  cat("page_meta      : limit =", pm$limit, "| offset =", pm$offset,
      "| total_count =", pm$total_count, "\n")
  first <- got$payload$activities[[1]]
  cat("fields per record:", length(first), "\n")
  for (f in c("activity_id", "molecule_chembl_id", "standard_type", "standard_relation",
              "standard_value", "standard_units", "assay_chembl_id"))
    cat(sprintf("   %-22s %s\n", f, if (is.null(first[[f]])) "NULL" else as.character(first[[f]])))
} else {
  cat("no live response; every number below comes from the frozen snapshot in ../data/\n")
}



The chunk printed its own provenance — `network`, `cache`, or a failure with its reason — which is the habit to copy: **a function that may or may not have reached the internet must say which happened**. Three details of the implementation are deliberate. `req_error(is_error = function(resp) FALSE)` switches off `httr2`'s default behaviour of raising on a 4xx or 5xx, so that *we* decide what to do with a status code instead of a condition propagating out of the chunk. `req_user_agent()` identifies the client, which is what a public service asks for and what lets an administrator contact you instead of blocking your institution. And the cache key is a hash of the URL *and* the parameters, so two different queries never collide. If the build machine had no connection, the message says so and the document continues on the frozen file, with every subsequent number still reproducible.

### 4.1 Pagination, and checking the live service against the frozen snapshot

The same query asked without a `limit` returns thousands of records in pages. The next chunk walks at most three pages, counts what the service reports, and compares that count with the frozen table shipped with this course — the single most informative check you can run on a downloaded dataset, because it tells you whether the database has moved since the snapshot was taken.



In [ ]:
## Pagination, capped, with the frozen snapshot as both fallback and reference.
frozen_ic50 <- sum(raw$standard_type == "IC50")
collected <- 0L; pages <- 0L; total_reported <- NA_integer_
url <- paste0(CHEMBL, "/activity.json")
params <- list(target_chembl_id = "CHEMBL3231", standard_type = "IC50", limit = 1000)

while (pages < 3L) {
  page <- api_get(url, params)
  if (is.null(page$payload)) {
    cat(sprintf("page %d: unavailable (%s) - stopping and using the frozen file\n",
                pages + 1L, page$provenance))
    break
  }
  pages <- pages + 1L
  collected <- collected + length(page$payload$activities)
  total_reported <- page$payload$page_meta$total_count
  cat(sprintf("page %d: %4d records  (%s); total reported by the service: %d\n",
              pages, length(page$payload$activities), page$provenance, total_reported))
  nxt <- page$payload$page_meta[["next"]]
  if (is.null(nxt)) break
  url <- paste0("https://www.ebi.ac.uk", nxt); params <- list()
}

cat("\nrecords collected in this session :", collected, "\n")
cat("IC50 records in the frozen snapshot:", frozen_ic50, "\n")
if (!is.na(total_reported)) {
  cat("live total vs frozen snapshot      :", total_reported, "vs", frozen_ic50, "->",
      if (total_reported == frozen_ic50) "unchanged"
      else sprintf("the database has moved by %+d records since the snapshot",
                   total_reported - frozen_ic50), "\n")
} else {
  cat("no live total available; the frozen snapshot is the reference for this document\n")
}



Read whichever message the chunk printed. If the live total equals the frozen count, the snapshot is still current and anything you compute from either source agrees. If it differs, you have just measured the phenomenon Unit 2 warned about — a moving database — and you now know by how much; note that a difference can also mean the two queries are not identical, since the frozen snapshot was built from a query over three measurement types and this page asks only for IC50. If the network was unavailable, the frozen file is the reference and the document is unaffected. In all three cases the analysis that follows is reproducible, because it reads `../data/`, and the live call served only to *date* the snapshot.

<div class="box warn"><h4>Terms of use are part of the method</h4>
<p>Public APIs are shared infrastructure. Identify your client with a user agent, respect published rate limits, cache aggressively, never parallelise a crawl across dozens of connections, and prefer the bulk download when you need more than a few thousand records — EBI and NCBI both publish full dumps precisely so that nobody has to page through millions of records. For production-scale work use the official clients and for the whole of ChEMBL use the official SQLite or PostgreSQL dump, which is one download and then local SQL — exactly the SQL of Section 6. This document deliberately does not download the dump; it is several gigabytes.</p></div>

## 5. Identifiers, and why mapping is the first thing that breaks

Each resource names its objects in its own way: `CHEMBL1201305`, PubChem `CID 2244`, UniProt `Q13464`, PDB `2ESM`, GEO `GSE16134`. Integrating two resources means mapping between two naming systems, and that mapping is the most error-prone step in a project, for four structural reasons.

- **Different granularity.** A PubChem SID is a *deposited substance* (one depositor's record, possibly a mixture or a salt); a CID is a *standardised compound*. A UniProt accession names a sequence entry; a PDB code names one experimental structure of (part of) it, and one protein may have dozens.
- **Many-to-one and one-to-many.** Several ChEMBL molecule ids collapse onto one standardised structure when salts and charge states are normalised; one UniProt accession maps to many PDB codes; one gene symbol maps to several probe sets on a microarray platform.
- **Structure-based identifiers are not free of pitfalls.** The **InChIKey** is a 27-character hash of the standard InChI: a 14-character skeleton block, a 10-character block for stereochemistry and isotopes, and a final character for the protonation state<sup class="c"><a href="#r22">[22]</a></sup>. Two stereoisomers therefore share the first block and differ in the second — which is a feature when you want "the same skeleton" and a bug when you match on the first block and silently merge enantiomers.
- **Identifiers are retired and merged.** Databases deprecate entries, and a mapping table downloaded last year may point at accessions that no longer exist. UniChem exists precisely to track cross-references between chemical structure identifiers across resources<sup class="c"><a href="#r23">[23]</a></sup>, and the error rates of public chemistry databases — wrong structures attached to right identifiers and vice versa — have been quantified repeatedly<sup class="c"><a href="#r31">[31]</a></sup>.

The Python edition demonstrates the stereochemistry trap by computing InChIKeys for two ibuprofen enantiomers with RDKit. Without RDKit we do something better: we measure the trap **in the real curated table**, by counting how many distinct structures share a skeleton block.



In [ ]:
## InChIKey anatomy, and the skeleton-block collision measured on the real curated table.
cur <- load_data("rock1_chembl_curated.csv")
keys <- data.table(inchikey = cur$inchikey,
                   skeleton = substr(cur$inchikey, 1, 14),
                   stereo = substr(cur$inchikey, 16, 25),
                   protonation = substr(cur$inchikey, 27, 27))
print(knitr::kable(head(keys, 4), row.names = FALSE, format = "simple"))

cat("\nstructures (distinct full InChIKeys):", uniqueN(keys$inchikey), "\n")
cat("distinct skeleton blocks            :", uniqueN(keys$skeleton), "\n")
coll <- keys[, .N, by = skeleton][N > 1][order(-N)]
cat("skeleton blocks carried by more than one structure:", nrow(coll),
    "| structures involved:", sum(coll$N), "\n")
cat(sprintf("deduplicating on the first 14 characters would merge %d structures into %d rows,\n",
            sum(coll$N), nrow(coll)))
cat(sprintf("i.e. it would silently lose %.1f%% of the compound table.\n",
            100 * (sum(coll$N) - nrow(coll)) / uniqueN(keys$inchikey)))
ex_sk <- coll$skeleton[1]
cat("\nexample collision (", ex_sk, "):\n", sep = "")
print(keys[skeleton == ex_sk, .(inchikey, stereo, protonation)])
cat("protonation characters present in the table:",
    paste(names(table(keys$protonation)), table(keys$protonation), sep = "=", collapse = "  "), "\n")



<p class="caption"><b>Table (Section 5).</b> The three blocks of an InChIKey, shown for the first four compounds of the curated table and then counted over all of it. The printed collision count is the quantitative version of the warning: in this one target's compound set, a non-trivial number of distinct structures share a skeleton block and differ only in stereochemistry or protonation. A pipeline that deduplicates on the first 14 characters — a shortcut that appears in published code — would average their activities into one meaningless number, and the example printed at the end shows a real instance.</p>

The curated table also carries the *other* kind of mapping: the `chembl_ids` column records every ChEMBL molecule identifier that collapsed onto one standardised parent structure. The next chunk measures how often that happens.



In [ ]:
## Many-to-one mapping in the real curated table: how many ChEMBL ids per standardised structure?
cur[, n_ids := lengths(strsplit(chembl_ids, "|", fixed = TRUE))]
counts <- cur[, .N, by = n_ids][order(n_ids)]
cat("compounds (unique InChIKeys):", nrow(cur), "\n")
cat("ChEMBL ids collapsed per structure:\n"); print(counts)
cat("\nstructures built from more than one ChEMBL id:", sum(cur$n_ids > 1),
    "| ids involved:", sum(cur$n_ids[cur$n_ids > 1]), "\n")
example <- cur[n_ids > 1][1]
cat("example:", example$chembl_ids, "->", example$inchikey, "\n")

p <- ggplot(counts, aes(factor(n_ids), N)) +
  geom_col(fill = PAL[["blue"]], width = 0.6) +
  geom_text(aes(label = format(N, big.mark = ",")), vjust = -0.4, size = 2.7) +
  scale_y_log10(expand = expansion(mult = c(0, 0.15))) +
  labs(x = "ChEMBL molecule ids collapsed onto one standardised structure",
       y = "number of structures (log scale)", title = "Identifier mapping is rarely one-to-one")
print(p)



<p class="caption"><b>Figure 5.</b> How many ChEMBL molecule identifiers collapse onto a single standardised structure in the curated ROCK1 table, on a logarithmic count axis. The overwhelming majority of structures come from exactly one identifier, but a non-trivial tail comes from two or more — salts, charge states and duplicate depositions of the same compound. Reporting "we studied <i>N</i> compounds" without saying whether <i>N</i> counts database identifiers or standardised structures is therefore ambiguous by a few per cent, and the curated table of this course counts structures.</p>

The practical danger of a many-to-one relationship is **row inflation**: joining two tables on a key that is not unique on either side multiplies rows. The next chunk produces the error deliberately on the real tables.



In [ ]:
## Row inflation: a join on a non-unique key, produced deliberately.
activities <- raw[, .(activity_id, molecule_chembl_id, standard_type, standard_value)]
parents <- cur[, .(inchikey, chembl_id = strsplit(chembl_ids, "|", fixed = TRUE))
               ][, .(chembl_id = unlist(chembl_id)), by = inchikey][, .(chembl_id, inchikey)]
cat("activities rows :", nrow(activities), "\n")
cat("parent map rows :", nrow(parents), "| unique chembl_id:", uniqueN(parents$chembl_id),
    "| unique inchikey:", uniqueN(parents$inchikey), "\n")

correct <- merge(activities, parents, by.x = "molecule_chembl_id", by.y = "chembl_id",
                 all.x = TRUE, sort = FALSE)
cat("\ncorrect join (map is unique on chembl_id):", nrow(correct), "rows ->",
    if (nrow(correct) == nrow(activities)) "unchanged" else "INFLATED", "\n")

## Now the error: join a second time on the structure, where the map is many-to-one.
inflated <- merge(correct, parents, by = "inchikey", allow.cartesian = TRUE, sort = FALSE)
cat("joining a second time on inchikey       :", nrow(inflated), "rows -> inflated by",
    sprintf("%+d (%.1f%%)", nrow(inflated) - nrow(activities),
            100 * nrow(inflated) / nrow(activities) - 100), "\n")
cat("\nmean standard_value before:", round(mean(activities$standard_value, na.rm = TRUE), 1),
    "| after the inflating join:", round(mean(inflated$standard_value, na.rm = TRUE), 1), "\n")
cat("the mean changed because duplicated rows are counted twice.\n")
cat("the guard:", tryCatch({ stopifnot(nrow(inflated) == nrow(activities)); "passed" },
                          error = function(e) "stopifnot() stops here, before the mean is reported"),
    "\n")



The row count grew and — the reason this matters — **the mean changed**, because compounds with several database identifiers now contribute several times to the average. No error was raised, the resulting table looks entirely normal, and every statistic computed from it is wrong by a few per cent in an unpredictable direction. Note one R-specific detail in the code: `data.table::merge` refused the inflating join until we passed `allow.cartesian = TRUE`, which is a guard rail the Python edition does not have — `pandas.merge` inflates silently unless you pass `validate="m:1"`. The defences are mechanical and you should apply all three: assert the expected row count after every merge, check `uniqueN(key) == nrow(tbl)` on the side that should be unique *before* merging, and leave `allow.cartesian` at its default so that `data.table` stops you.

<div class="box key"><h4>The identifier rules of this group</h4>
<p>Store the <em>structure-based</em> identifier (InChIKey) as the row key of a compound table, keep the database identifiers as a separate column, and never deduplicate on the skeleton block alone — Section 5 measured what that would cost on this very table. For proteins store the UniProt accession, not the gene symbol: symbols are ambiguous across species and change over time. Record the <em>version</em> of every mapping table you use. And whenever a join is performed, state in the code what cardinality you expect and let the tool verify it.</p></div>

## 6. SQL in one hour

A folder of CSV files is a database with no schema, no types, no constraints, no indexes and no query language. A **relational database** supplies all five. The relational model<sup class="c"><a href="#r24">[24]</a></sup> is simple enough to state in three sentences: data live in *tables* (relations) of *rows* (tuples) with named, typed *columns*; one column (or a combination) is the **primary key**, unique per row; a column referring to another table's key is a **foreign key**. Every query is then a combination of selecting rows, choosing columns, joining tables on keys, and grouping.

**SQLite** is the right database for a thesis project: a single file, no server, and one of the most widely deployed pieces of software in existence<sup class="c"><a href="#r25">[25]</a></sup>. R reaches it through **DBI**, the common database interface, with `RSQLite` as the driver — the same `DBI` calls work against PostgreSQL or DuckDB by changing one line. ChEMBL itself is distributed as a SQLite dump, so the SQL you write here is the SQL you would run against the whole database. We build a three-table database from the frozen course files.



In [ ]:
## Build an in-document SQLite database from the frozen CSVs: activities, compounds, assays.
con <- dbConnect(SQLite(), file.path(SCRATCH, "rock1.sqlite"))

activities_tbl <- raw[, .(activity_id, molecule_chembl_id, assay_chembl_id, standard_type,
                          standard_relation, standard_value, standard_units, pchembl_value)]
compounds_tbl <- unique(raw[, .(molecule_chembl_id, canonical_smiles, pref_name, max_phase,
                                molecule_type)], by = "molecule_chembl_id")
assays_tbl <- unique(raw[, .(assay_chembl_id, assay_type, assay_description,
                             document_chembl_id, bao_label)], by = "assay_chembl_id")

dbWriteTable(con, "activities", activities_tbl, overwrite = TRUE)
dbWriteTable(con, "compounds", compounds_tbl, overwrite = TRUE)
dbWriteTable(con, "assays", assays_tbl, overwrite = TRUE)

cat("tables and row counts:\n")
for (nm in c("activities", "compounds", "assays")) {
  n <- dbGetQuery(con, sprintf("SELECT COUNT(*) AS n FROM %s", nm))$n
  cols <- dbListFields(con, nm)
  cat(sprintf("  %-11s %5d rows | columns: %s\n", nm, n, paste(cols, collapse = ", ")))
}
cat(sprintf("\ndatabase file size: %.2f MB (the CSV it came from is %.2f MB)\n",
            file.size(file.path(SCRATCH, "rock1.sqlite")) / 1e6,
            file.size(data_path("rock1_chembl_raw.csv")) / 1e6))
cat("primary keys here are activity_id, molecule_chembl_id and assay_chembl_id;\n")
cat("activities.molecule_chembl_id and activities.assay_chembl_id are the foreign keys.\n")



Three tables, one row per thing, no repetition: the assay description appears once in `assays` instead of once per measurement, which is what **normalisation** means and why the database file is smaller than the CSV it came from. The next chunk is the SQL vocabulary in one pass — `SELECT` which columns, `FROM` which table, `WHERE` which rows, `ORDER BY` in which order, `LIMIT` how many.



In [ ]:
## SELECT / WHERE / ORDER BY / LIMIT: the ten most potent exactly-measured IC50 records.
q1 <- "
SELECT a.molecule_chembl_id, c.pref_name, a.standard_value AS ic50_nM, a.assay_chembl_id
FROM activities AS a
JOIN compounds  AS c ON c.molecule_chembl_id = a.molecule_chembl_id
WHERE a.standard_type = 'IC50'
  AND a.standard_relation = '='
  AND a.standard_value IS NOT NULL
ORDER BY a.standard_value ASC
LIMIT 10"
top10 <- dbGetQuery(con, q1)
print(knitr::kable(top10, row.names = FALSE, format = "simple"))
cat(sprintf("\nmost potent recorded IC50: %.3f nM | named compounds among the top 10: %d\n",
            min(top10$ic50_nM), sum(nzchar(top10$pref_name))))
cat("records with standard_value exactly zero in the whole table:",
    nrow(dbGetQuery(con, "SELECT 1 FROM activities WHERE standard_value = 0")), "\n")



<p class="caption"><b>Table (Section 6).</b> The ten most potent exactly-measured IC50 records in the frozen snapshot, obtained with one SQL statement that joins the activity table to the compound table. Note the <code>IS NOT NULL</code> condition: in SQL, <code>NULL</code> is not a value but the absence of one, so <code>standard_value = NULL</code> is never true — comparisons with <code>NULL</code> yield <code>NULL</code>, which is not <code>TRUE</code>. This is the single most common SQL mistake among newcomers.</p>

Read the first row of that table before you celebrate. The smallest "IC50" in the snapshot is a value of zero, which is not a potency but a defect — a record that ChEMBL itself flags in its `data_validity_comment` column as being outside the typical range, and the printed count tells you how many such records the table holds. A concentration of zero has no logarithm, so it would propagate as an infinity into every pActivity computed from it. This is the raw table doing its job as a teaching dataset, and it illustrates the rule that a query is only as trustworthy as its filters: the honest version of this question adds `AND a.standard_value > 0` and excludes flagged records, which is exactly what the reference curation protocol does before the curated table is produced. Quality problems of this kind are endemic to public chemistry databases rather than peculiar to this target<sup class="c"><a href="#r31">[31]</a></sup>.

### 6.1 A worked example by hand: inner join, left join, and what `COUNT` counts

Joins are best learned on tables small enough to trace with a pencil. Take three compounds and three measurements:

**`compounds`**

| compound_id | name |
|---|---|
| C1 | fasudil |
| C2 | Y-27632 |
| C3 | ripasudil |

**`activities`**

| activity_id | compound_id | ic50_nM |
|---|---|---|
| A1 | C1 | 25 |
| A2 | C1 | 30 |
| A3 | C2 | 500 |

Tracing by hand:

- **`INNER JOIN` on `compound_id`** keeps only pairs that exist on both sides: A1–C1, A2–C1, A3–C2 → **3 rows**. C3 disappears, because it has no measurement. If you then count compounds in the result you get **2**, not 3 — the inner join silently dropped the untested compound.
- **`LEFT JOIN` from `compounds`** keeps every compound and fills the missing side with `NULL`: C1–A1, C1–A2, C2–A3, C3–`NULL` → **4 rows**.
- On that left join, `COUNT(*)` **per compound** gives C1 → 2, C2 → 1, C3 → **1**, because the row for C3 exists even though its measurement is `NULL`. But `COUNT(activity_id)` gives C1 → 2, C2 → 1, C3 → **0**, because `COUNT(column)` ignores `NULL`s. Only the second is the answer to "how many measurements does this compound have".

That last distinction is the trap: a query that reports "every compound has at least one measurement" is usually a `COUNT(*)` over a left join. The chunk below confirms all three results in SQL and reproduces them with `dplyr`.



In [ ]:
## Confirming the hand-traced joins in SQL, and reproducing them in dplyr.
small_c <- data.frame(compound_id = c("C1", "C2", "C3"),
                      name = c("fasudil", "Y-27632", "ripasudil"))
small_a <- data.frame(activity_id = c("A1", "A2", "A3"),
                      compound_id = c("C1", "C1", "C2"), ic50_nM = c(25, 30, 500))
dbWriteTable(con, "small_compounds", small_c, overwrite = TRUE)
dbWriteTable(con, "small_activities", small_a, overwrite = TRUE)

inner <- dbGetQuery(con, "SELECT c.compound_id, c.name, a.activity_id, a.ic50_nM
                          FROM small_compounds c JOIN small_activities a USING (compound_id)")
left  <- dbGetQuery(con, "SELECT c.compound_id, c.name, a.activity_id, a.ic50_nM
                          FROM small_compounds c LEFT JOIN small_activities a USING (compound_id)")
counts_sql <- dbGetQuery(con, "SELECT c.compound_id, COUNT(*) AS count_star,
                                      COUNT(a.activity_id) AS count_col
                               FROM small_compounds c
                               LEFT JOIN small_activities a USING (compound_id)
                               GROUP BY c.compound_id ORDER BY c.compound_id")

cat("INNER JOIN ->", nrow(inner), "rows (hand-traced: 3); compounds present:",
    length(unique(inner$compound_id)), "(hand-traced: 2)\n")
cat("LEFT  JOIN ->", nrow(left), "rows (hand-traced: 4)\n\n")
cat("COUNT(*) versus COUNT(activity_id) per compound:\n")
print(knitr::kable(counts_sql, row.names = FALSE, format = "simple"))

counts_dplyr <- small_c |>
  left_join(small_a, by = "compound_id") |>
  group_by(compound_id) |>
  summarise(count_star = n(), count_col = sum(!is.na(activity_id)), .groups = "drop") |>
  as.data.frame()
cat("\ndplyr reproduces the same table:",
    isTRUE(all.equal(counts_dplyr, counts_sql, check.attributes = FALSE)), "\n")
cat("hand = SQL = dplyr:", identical(counts_sql$count_col, c(2L, 1L, 0L)) &&
    identical(as.integer(counts_dplyr$count_col), c(2L, 1L, 0L)), "\n")



All three agree with the pencil: three rows from the inner join, four from the left join, and `COUNT(activity_id)` giving zero for the untested compound where `COUNT(*)` gives one. Note the `dplyr` correspondence, which is worth memorising because it makes SQL readable to a `dplyr` user and vice versa: `WHERE` is `filter()`, `JOIN` is `left_join()`/`inner_join()`, `GROUP BY ... agg` is `group_by() |> summarise()`, `ORDER BY` is `arrange()`, `LIMIT` is `slice_head()`, `COUNT(*)` is `n()` and `COUNT(col)` is `sum(!is.na(col))`. The correspondence is not accidental — `dplyr`'s verbs were designed as a grammar over the relational operations<sup class="c"><a href="#r34">[34]</a></sup>, which is why `dbplyr` can translate the same pipeline into SQL and run it inside the database.



In [ ]:
## GROUP BY and HAVING on the real tables: which assay types carry the data?
q2 <- "
SELECT s.assay_type,
       s.bao_label,
       COUNT(*)                             AS n_records,
       COUNT(DISTINCT a.molecule_chembl_id) AS n_compounds,
       COUNT(DISTINCT s.document_chembl_id) AS n_documents,
       ROUND(AVG(a.standard_value), 1)      AS mean_nM,
       SUM(CASE WHEN a.standard_relation = '=' THEN 1 ELSE 0 END) AS n_exact
FROM activities AS a
JOIN assays     AS s ON s.assay_chembl_id = a.assay_chembl_id
GROUP BY s.assay_type, s.bao_label
HAVING COUNT(*) >= 10
ORDER BY n_records DESC"
by_assay <- dbGetQuery(con, q2)
print(knitr::kable(by_assay, row.names = FALSE, format = "simple"))

by_assay$label <- factor(paste(by_assay$assay_type, "-", by_assay$bao_label),
                         levels = rev(paste(by_assay$assay_type, "-", by_assay$bao_label)))
p <- ggplot(by_assay) +
  geom_col(aes(n_records, label, fill = "records"), width = 0.65) +
  geom_col(aes(n_exact, label, fill = "of which exactly measured"), width = 0.65) +
  geom_text(aes(n_records, label,
                label = sprintf(" %s (%.0f%% exact)", format(n_records, big.mark = ","),
                                100 * n_exact / n_records)), hjust = 0, size = 2.5) +
  scale_fill_manual(values = c(records = PAL[["blue"]],
                               `of which exactly measured` = "#a8c6e6")) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.3))) +
  labs(x = "activity records", y = NULL, fill = NULL,
       title = "Where the ROCK1 measurements come from") +
  theme(legend.position = "bottom", axis.text.y = element_text(size = 7))
print(p)



<p class="caption"><b>Figure 6.</b> Activity records per assay type and BioAssay Ontology label, from the <code>GROUP BY … HAVING</code> query above, with the exactly-measured subset shown in the lighter shade and its percentage annotated. Binding assays (<code>B</code>) and functional assays (<code>F</code>) measure different things — displacement of a ligand versus inhibition of enzymatic activity — and mixing them in one regression target is a modelling decision that must be made explicitly, not inherited from a download. The <code>HAVING</code> clause filtered out the long tail of assay categories with fewer than ten records; note that <code>HAVING</code> filters <em>groups</em> after aggregation, whereas <code>WHERE</code> filters rows before it.</p>

### 6.2 Subqueries, and the question a folder of CSVs cannot answer

A **subquery** is a query used as a table. The pattern below — find the compounds measured in at least three distinct documents, then report their activity spread — is the kind of question that requires two passes over the data and is natural in SQL.



In [ ]:
## A subquery: compounds measured in >= 3 distinct documents, with their IC50 spread.
q3 <- "
SELECT m.molecule_chembl_id, c.pref_name, m.n_documents, m.n_records,
       ROUND(m.min_nM, 1) AS min_nM, ROUND(m.max_nM, 1) AS max_nM,
       ROUND(m.max_nM / m.min_nM, 1) AS fold_spread
FROM (
    SELECT a.molecule_chembl_id,
           COUNT(DISTINCT s.document_chembl_id) AS n_documents,
           COUNT(*)                             AS n_records,
           MIN(a.standard_value)                AS min_nM,
           MAX(a.standard_value)                AS max_nM
    FROM activities a
    JOIN assays s ON s.assay_chembl_id = a.assay_chembl_id
    WHERE a.standard_type = 'IC50' AND a.standard_relation = '=' AND a.standard_value > 0
    GROUP BY a.molecule_chembl_id
) AS m
JOIN compounds c ON c.molecule_chembl_id = m.molecule_chembl_id
WHERE m.n_documents >= 3
ORDER BY fold_spread DESC
LIMIT 8"
spread <- dbGetQuery(con, q3)
print(knitr::kable(spread, row.names = FALSE, format = "simple"))
cat(sprintf("\nlargest fold spread between two published IC50 values for the same compound: %.0f-fold\n",
            max(spread$fold_spread)))
cat(sprintf("that is %.1f log units - larger than the effect most QSAR models are asked to predict.\n",
            log10(max(spread$fold_spread))))



<p class="caption"><b>Table (Section 6.2).</b> Compounds whose IC50 was measured in at least three distinct publications, ordered by the ratio between their largest and smallest reported value. The printed fold spread is the experimental reality this course keeps returning to: independent laboratories measuring the same compound against the same target disagree, sometimes by more than an order of magnitude<sup class="c"><a href="#r26">[26]</a></sup><sup class="c"><a href="#r27">[27]</a></sup><sup class="c"><a href="#r28">[28]</a></sup>. Units 6 and 7 decide what to do about it — aggregate with a median, keep the spread as a column, and never report a model error smaller than the data's own noise.</p>

### 6.3 Indexes: why a database beats a folder

An **index** is a sorted structure that lets the database find rows by a key without scanning the table. Creating one costs disk and slows writes; it changes a linear scan into a logarithmic lookup. The next chunk measures it on a table of about a hundred thousand rows.



In [ ]:
## The cost of a scan versus the cost of an index, measured.
wide_tbl <- rbindlist(rep(list(activities_tbl), 20))
dbWriteTable(con, "activities_big", wide_tbl, overwrite = TRUE)
set.seed(0)
probe_ids <- sample(unique(raw$molecule_chembl_id[nzchar(raw$molecule_chembl_id)]), 40)

timed_lookups <- function(ids) {
  t0 <- proc.time()[["elapsed"]]
  for (mid in ids)
    dbGetQuery(con, "SELECT COUNT(*) FROM activities_big WHERE molecule_chembl_id = ?",
               params = list(mid))
  proc.time()[["elapsed"]] - t0
}

no_index <- timed_lookups(probe_ids)
t0 <- proc.time()[["elapsed"]]
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_mol ON activities_big(molecule_chembl_id)")
build <- proc.time()[["elapsed"]] - t0
with_index <- timed_lookups(probe_ids)

cat(sprintf("table: %s rows\n", format(nrow(wide_tbl), big.mark = ",")))
cat(sprintf("40 lookups without an index: %8.1f ms  (%.2f ms each)\n",
            no_index * 1000, no_index / 40 * 1000))
cat(sprintf("building the index         : %8.1f ms\n", build * 1000))
cat(sprintf("40 lookups with the index  : %8.1f ms  (%.2f ms each)\n",
            with_index * 1000, with_index / 40 * 1000))
cat(sprintf("speed-up: %.0fx; the index paid for itself after %.0f lookups\n",
            no_index / with_index, build / ((no_index - with_index) / 40)))

d <- data.frame(stage = factor(c("full scan\n(no index)", "index build\n(once)", "indexed lookup"),
                               levels = c("full scan\n(no index)", "index build\n(once)",
                                          "indexed lookup")),
                ms = c(no_index, build, with_index) * 1000)
p <- ggplot(d, aes(stage, ms, fill = stage)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.0f ms", ms)), vjust = -0.4, size = 2.9, fontface = "bold") +
  scale_fill_manual(values = c(PAL[["orange"]], "grey60", PAL[["blue"]])) +
  scale_y_log10(expand = expansion(mult = c(0, 0.18))) +
  labs(x = NULL, y = "milliseconds (log scale)",
       title = sprintf("40 key lookups in a %s-row table", format(nrow(wide_tbl), big.mark = ",")))
print(p)



<p class="caption"><b>Figure 7.</b> Time for forty key lookups in a table of about a hundred thousand rows, before and after creating an index on the lookup column, with the one-off index build shown in grey (logarithmic axis). The measured speed-up and the break-even point are printed above the figure: the index repays its construction cost after a handful of queries. This is the concrete answer to "when does a database beat a folder of CSV files?" — when you query the same data repeatedly by key, when it does not fit in memory, when several processes read it at once, or when you need the types and constraints a schema enforces. For a single pass over a table that fits in RAM, <code>data.table</code> is simpler and faster, and its own <code>setkey()</code> is the in-memory equivalent of the index measured here.</p>

### 6.4 The same question, twice

The final SQL exercise is the one the syllabus asks for: answer one question in both systems and compare the answers value by value.



In [ ]:
## One question, two implementations: 'per measurement type, how many exact records and what spread?'
question_sql <- "
SELECT standard_type,
       COUNT(*) AS n,
       SUM(CASE WHEN standard_relation = '=' THEN 1 ELSE 0 END) AS n_exact,
       ROUND(AVG(standard_value), 3) AS mean_nM,
       ROUND(MIN(standard_value), 3) AS min_nM,
       ROUND(MAX(standard_value), 3) AS max_nM
FROM activities
WHERE standard_value IS NOT NULL
GROUP BY standard_type
ORDER BY n DESC"
t0 <- proc.time()[["elapsed"]]; sql_answer <- dbGetQuery(con, question_sql)
t_sql <- proc.time()[["elapsed"]] - t0

t0 <- proc.time()[["elapsed"]]
dplyr_answer <- activities_tbl |>
  filter(!is.na(standard_value)) |>
  group_by(standard_type) |>
  summarise(n = n(), n_exact = sum(standard_relation == "="),
            mean_nM = round(mean(standard_value), 3), min_nM = round(min(standard_value), 3),
            max_nM = round(max(standard_value), 3), .groups = "drop") |>
  arrange(desc(n)) |> as.data.frame()
t_dplyr <- proc.time()[["elapsed"]] - t0

cat("SQL:\n"); print(knitr::kable(sql_answer, row.names = FALSE, format = "simple"))
cat("\ndplyr:\n"); print(knitr::kable(dplyr_answer, row.names = FALSE, format = "simple"))
num <- c("n", "n_exact", "mean_nM", "min_nM", "max_nM")
cat("\nidentical shape:", identical(dim(sql_answer), dim(dplyr_answer)),
    "| identical keys:", identical(sql_answer$standard_type, dplyr_answer$standard_type),
    "| identical values:",
    all(abs(as.matrix(sql_answer[num]) - as.matrix(dplyr_answer[num])) < 1e-3), "\n")
cat(sprintf("timing on this table: SQL %.1f ms vs dplyr %.1f ms\n", t_sql * 1000, t_dplyr * 1000))
cat("maximum absolute difference across the compared columns:",
    max(abs(as.matrix(sql_answer[num]) - as.matrix(dplyr_answer[num]))), "\n")

cmp <- rbind(data.frame(type = sql_answer$standard_type, n = sql_answer$n, system = "SQL"),
             data.frame(type = dplyr_answer$standard_type, n = dplyr_answer$n, system = "dplyr"))
p1 <- ggplot(cmp, aes(type, n, fill = system)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.65) +
  geom_text(aes(label = format(n, big.mark = ",")), position = position_dodge(width = 0.75),
            vjust = -0.3, size = 2.5) +
  scale_fill_manual(values = c(SQL = PAL[["blue"]], dplyr = "#a8c6e6")) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.14))) +
  labs(x = NULL, y = "records with a value", fill = NULL, title = "Same counts from both systems") +
  theme(legend.position = "bottom")
p2 <- ggplot(data.frame(system = c("SQL", "dplyr"), ms = c(t_sql, t_dplyr) * 1000),
             aes(system, ms, fill = system)) +
  geom_col(width = 0.5, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.1f ms", ms)), vjust = -0.3, size = 2.9, fontface = "bold") +
  scale_fill_manual(values = c(SQL = PAL[["blue"]], dplyr = "#a8c6e6")) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.16))) +
  labs(x = NULL, y = "time for the whole query (ms)", title = "and comparable cost on this table")
print(p1 | p2)



<p class="caption"><b>Figure 8.</b> Left: the record count per measurement type as computed by SQL (dark) and by <code>dplyr</code> (light) — the bars are identical, which is the verification the syllabus asks for, and the printed maximum absolute difference across the compared columns is zero to rounding. Right: the wall-clock cost of each implementation on this in-memory-sized table. Neither system is "the fast one"; what differs is where the data already are and how the question is phrased.</p>

The two answers agree on every key and every value to three decimals — which is the point of the exercise. SQL and `dplyr` are two syntaxes for the same relational operations, and the choice between them is pragmatic: SQL when the data live in a database, when the query joins several tables, or when the table does not fit in memory; `dplyr` when the data are already loaded, when the operation is not expressible relationally (a bootstrap, a model fit, a plot), or when it is one step of a longer R pipeline. The timing printed above is for one table that fits comfortably in memory, where the difference is of no consequence — and a measurement, not a general claim. R offers a third option worth knowing: `dbplyr` lets you write `dplyr` verbs against a database connection and translates them to SQL, so the same pipeline can run in R or in the database by changing where the data come from.

<div class="exercise"><h4>Exercise 2</h4>
<p>Write the SQL that answers: "for each assay type, how many distinct compounds have at least two exactly-measured IC50 records in different documents?" You will need a subquery, a <code>GROUP BY</code> with <code>HAVING</code>, and a join to <code>assays</code>. Then write the <code>dplyr</code> equivalent and compare the two results as above. Finally, add an index on <code>activities(standard_type, standard_relation)</code> and re-time your query: does it help, and why does an index on a column with three distinct values help less than an index on a compound identifier? For the last part, run the same pipeline through <code>dbplyr</code> (<code>tbl(con, "activities") |&gt; group_by(...) |&gt; summarise(...)</code>) and inspect the generated SQL with <code>show_query()</code>.</p></div>

## 7. Terms of use, licences and when scraping is not acceptable

Three obligations attach to data you did not generate, and all three are part of the method rather than of the paperwork.

**The licence governs redistribution.** ChEMBL is CC BY-SA 3.0, so a derived table must carry the same licence (Unit 2); PubChem's own content is public domain, but individual depositor submissions may carry their own terms; the PDB is open; GEO data are public but the associated publications are not. Record the licence of every source in your manifest before building a derived table.

**The terms of use govern access.** Public APIs publish rate limits and acceptable-use policies. NCBI, for instance, limits anonymous E-utilities requests and asks for an identifying tool name and contact address — which is why the client in Section 4 sets a user agent. Exceeding the limits gets your institution's address blocked, which affects your colleagues, not only you.

**`robots.txt` governs crawling.** A web server publishes `/robots.txt` to declare which paths automated clients may fetch; the format is standardised as RFC 9309<sup class="c"><a href="#r29">[29]</a></sup>. R has no parser for it in the environment that builds these pages — Python's standard library has one, which the Python edition uses — so the next chunk implements the subset that matters in a dozen lines, which is itself instructive: the format is small enough that there is no excuse for not checking.



In [ ]:
## Parsing a robots.txt policy (text written in-chunk, not fetched) and honouring it.
policy <- c("User-agent: *", "Disallow: /private/", "Disallow: /cgi-bin/", "Crawl-delay: 5",
            "", "User-agent: GreedyBot", "Disallow: /")
writeLines(policy, file.path(SCRATCH, "robots.txt"))

parse_robots <- function(lines) {
  groups <- list(); agents <- character(0); rules <- list(); delay <- NA_real_
  flush <- function() if (length(agents))
    for (a in agents) groups[[a]] <<- list(rules = rules, delay = delay)
  for (ln in c(lines, "")) {
    ln <- trimws(sub("#.*", "", ln))
    if (!nzchar(ln)) { flush(); agents <- character(0); rules <- list(); delay <- NA_real_; next }
    kv <- strsplit(ln, ":", fixed = TRUE)[[1]]
    key <- tolower(trimws(kv[1])); value <- trimws(paste(kv[-1], collapse = ":"))
    if (key == "user-agent") agents <- c(agents, value)
    else if (key %in% c("disallow", "allow")) rules[[length(rules) + 1]] <- list(key, value)
    else if (key == "crawl-delay") delay <- as.numeric(value)
  }
  flush(); groups
}
can_fetch <- function(groups, agent, path) {
  g <- groups[[agent]]; if (is.null(g)) g <- groups[["*"]]
  if (is.null(g)) return(TRUE)
  allowed <- TRUE
  for (r in g$rules) {
    if (nzchar(r[[2]]) && startsWith(path, r[[2]])) allowed <- (r[[1]] == "allow")
  }
  allowed
}

groups <- parse_robots(policy)
tests <- list(c("*", "/data/activities.csv"), c("*", "/private/internal.csv"),
              c("*", "/cgi-bin/query"), c("GreedyBot", "/data/activities.csv"))
for (t in tests)
  cat(sprintf("  %-10s https://example.org%-30s -> %s\n", t[1], t[2],
              if (can_fetch(groups, t[1], t[2])) "allowed" else "DISALLOWED"))
cat("\ncrawl delay requested for a generic client:", groups[["*"]]$delay, "seconds\n")
cat("groups parsed:", paste(names(groups), collapse = ", "), "\n")
cat("\nAnd the rule no file can express: if a bulk download or an API exists, use it instead.\n")



The parser answers per user agent and per path, and it reports the requested crawl delay — which a polite client sleeps for between requests. Our twelve-line version implements prefix matching and "last matching rule wins"; the real standard adds wildcards, `$` anchoring and a longest-match rule, so use a maintained implementation in production. But the decisive judgement is not technical. Scraping is **not acceptable** when an API or a bulk download exists (you are generating load for no reason), when the terms of use forbid it, when the content is behind authentication or a paywall, when the data are personal, or when your crawl would degrade a service other researchers depend on. In this laboratory, the order of preference is always: bulk download, then official API client, then hand-written API calls, and scraping only for a page that exists nowhere else — after asking.



In [ ]:
## Close the database and remove every temporary artefact.
dbDisconnect(con)
unlink(SCRATCH, recursive = TRUE, force = TRUE)
cat("scratch directory removed:", !dir.exists(SCRATCH), "\n")
cat("the frozen tables in ../data/ were read, never modified:",
    file.exists(data_path("rock1_chembl_raw.csv")), "\n")



<div class="box bridge"><h4>Bridge — where the data go next</h4>
<p>This unit filled <code>data/raw/</code>. <a href="../U2_Reproducibility/index.html">Unit 2</a> is what makes its downloads citable: manifest, checksum, licence, DOI; <a href="../U1_Toolkit/index.html">Unit 1</a> built the project layout they land in. The later units of this R edition follow the sequence of the <a href="../../Python/index.html">Python edition</a>, whose counterparts already exist: <a href="../U4_Tidy_Data/index.html">tidy data</a> reshapes what arrived into a tidy table<sup class="c"><a href="#r30">[30]</a></sup> — one row per observation, one column per variable — and <a href="../U6_Data_Quality/index.html">data quality</a> curates it into the modelling table used from Unit 7 onwards, with a logged decision for every record it removes. <a href="../U21_Omics/index.html">Sequence and omics data</a> returns to FASTA and GEO, <a href="../U20_MD_Analytics/index.html">molecular-dynamics analytics</a> to trajectory formats, and <a href="../U24_Software_Engineering/index.html">software engineering</a> turns the API client of Section 4 into a tested, installable package. In the sister courses, the cheminformatics elective covers <a href="../../../Cheminformatics/U3_SMILES_InChI_Formats/index.html">SMILES and InChI</a> and <a href="../../../Cheminformatics/U6_Databases/index.html">chemical databases</a> in chemical depth, while BGM324 shows what is done with the structures retrieved from the PDB in <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">structure prediction</a> and <a href="../../../BGM324/03_Molecular_Docking/index.html">docking</a>, and the R edition of BGM411 (<a href="../../../BGM411/R/index.html">study material</a>) reads the same tables into its model code.</p></div>

## 8. Summary

You should now be able to explain, without notes:

- The four axes along which file formats differ, and which format this group uses for raw downloads, intermediate tables, large numeric arrays and nested metadata — with the measurement that justifies each choice, and why RDS is right for a fitted model and wrong for an exchanged table.
- Exactly what a CSV loses, which kinds of value do not survive a round trip in R, and the three `fread` arguments that defend against it — including the one whose *default* destroys a column containing the literal text `NA`.
- Why a columnar format reads the same table an order of magnitude faster than a CSV, and why `fread`'s `select =` nevertheless gains proportionally as much — it skips column construction, not the scan.
- What SMILES, SDF, FASTA and PDB/mmCIF each store, how a fixed-column format is parsed with `substr()`, and why splitting a PDB line on whitespace is a bug that passes its first test.
- The anatomy of an HTTP request, the meaning of the status codes that matter, and the four disciplines of a well-behaved client: timeout, pagination, backoff with jitter, cache — plus the user agent that lets an administrator contact you instead of blocking you.
- Why a live call must always print its own provenance, and why this course's documents read frozen files.
- The identifier systems of the field, the three blocks of an InChIKey, and how many structures in this course's own table would be merged by deduplicating on the skeleton block.
- What row inflation is, how a non-unique join key changes a mean, and the three defences against it — one of which `data.table` enforces for you.
- The relational model in three sentences; `SELECT`, `WHERE`, `JOIN`, `GROUP BY`, `HAVING`, subqueries; why `NULL` comparisons are never true; and the difference between `COUNT(*)` and `COUNT(column)` on a left join.
- When a database beats a folder of CSV files, what an index costs and buys, and what `setkey()` is the in-memory equivalent of.
- The licence, terms-of-use and `robots.txt` obligations, and the order of preference: bulk download, official client, hand-written calls, and scraping only as a last resort after asking.

## 9. Self-assessment

<details><summary>Q1. A collaborator sends a CSV in which a compound identifier column has lost its leading zeros and a comment column has become empty. Explain both, and give the <code>fread</code> call that would have prevented them.</summary>
<p>CSV stores characters, not types, so the reader inferred an integer type for the identifier column (dropping leading zeros) and treated the literal string <code>NA</code> as missing, because <code>fread</code>'s default <code>na.strings</code> includes it. Prevent both with explicit arguments: <code>fread(path, colClasses = c(plate_well = "character"), na.strings = "")</code>. Better still, exchange the file as Parquet, which carries a schema and round-trips exactly.</p></details>

<details><summary>Q2. Restricting a read to two of twenty-five columns speeds up both <code>fread</code> and <code>read_parquet</code>. Does that mean the columnar layout buys nothing?</summary>
<p>No — it means the two readers save different things. Parquet stores each column as a separate compressed block with its own metadata, so it seeks to two blocks and never touches the rest of the file; its absolute times are an order of magnitude lower than the CSV reader's, on a file a hundred times smaller. <code>fread</code> must still scan every byte to locate the separators, but that multi-threaded scan is not where its time goes: allocating and type-converting twenty-five columns, several of them long strings, is, and <code>select =</code> skips exactly that. Hence a similar <em>ratio</em> from a much worse <em>baseline</em>. The measured times are printed in Section 2.</p></details>

<details><summary>Q3. You must store a 2048-bit fingerprint for ten million molecules. Which format, and why not CSV?</summary>
<p>A binary array format — HDF5, or a sparse matrix written as Parquet — because the data are a homogeneous numeric matrix, and because a CSV would store each bit as a character plus a separator, inflating the file by an order of magnitude and making every read a parse. A sparse representation is better still when most bits are zero, which is why this course's frozen fingerprint files store only the set bits as (id, bit) pairs.</p></details>

<details><summary>Q4. Which HTTP status codes should a client retry, which should it never retry, and why does the backoff need jitter?</summary>
<p>Retry 429 and 5xx (transient: rate limit or server failure). Never retry 400, 401, 403 or 404 — the request itself is wrong, so repeating it only adds load. Jitter decorrelates the retries of many clients that failed at the same moment, preventing a synchronised second wave that would knock the service over again.</p></details>

<details><summary>Q5. This document calls a live API but its results are reproducible. How?</summary>
<p>Because the live call is wrapped in <code>tryCatch()</code> with a timeout, its result is cached, its provenance is printed, and every number used downstream comes from the frozen, checksummed file in <code>../data/</code>. The network call serves to <em>date</em> the snapshot — comparing the live record count with the frozen one — not to produce the analysis. Note also <code>req_error(is_error = function(resp) FALSE)</code>, which stops <code>httr2</code> from raising on a 4xx or 5xx so that the chunk can report the status rather than abort the knit.</p></details>

<details><summary>Q6. Two molecules share the first 14 characters of their InChIKey. What do you know, and what must you not conclude?</summary>
<p>You know they have the same constitutional skeleton — the same atoms connected in the same way. You must not conclude they are the same compound: the second block encodes stereochemistry and isotopes, so enantiomers and diastereomers share the skeleton block while differing pharmacologically, and the final character encodes protonation state. Section 5 counted how many such collisions exist in this course's own curated table; match on the full key.</p></details>

<details><summary>Q7. After a merge, your table has more rows than the left table and the mean of a value column has changed. What happened, and what are the three defences?</summary>
<p>The join key was not unique on the right-hand side, so matching rows were duplicated (row inflation), and duplicated rows now contribute multiple times to the mean. Defences: assert the row count after the merge; check <code>uniqueN(key) == nrow(tbl)</code> on the side that must be unique before merging; and leave <code>data.table</code>'s <code>allow.cartesian = FALSE</code> default in place so that it stops you. In <code>dplyr</code>, <code>relationship = "many-to-one"</code> plays the same role.</p></details>

<details><summary>Q8. Why does <code>WHERE standard_value = NULL</code> return no rows, and what is the correct clause?</summary>
<p>Because <code>NULL</code> represents the absence of a value, and any comparison with it evaluates to <code>NULL</code>, which is not <code>TRUE</code>, so no row satisfies the condition. Use <code>IS NULL</code> or <code>IS NOT NULL</code>. The same logic explains why <code>NULL</code> rows vanish from inner joins and why aggregate functions ignore them — and it is the same reason R requires <code>is.na(x)</code> rather than <code>x == NA</code>.</p></details>

<details><summary>Q9. On a left join of compounds to activities, <code>COUNT(*)</code> reports 1 for a compound with no measurements. Why, and which function should you use?</summary>
<p>The left join produced one row for that compound with <code>NULL</code> in the activity columns, and <code>COUNT(*)</code> counts rows regardless of content. <code>COUNT(activity_id)</code> ignores <code>NULL</code>s and correctly reports 0. This is the hand-traced result of Section 6.1, confirmed in SQL and in <code>dplyr</code>, where the equivalent pair is <code>n()</code> and <code>sum(!is.na(activity_id))</code>.</p></details>

<details><summary>Q10. When is a database the right tool, and when is <code>data.table</code>?</summary>
<p>A database when the data are queried repeatedly by key, when they exceed memory, when several processes read them concurrently, or when types and constraints must be enforced; an index then turns scans into lookups (Figure 7). <code>data.table</code> when the table fits in memory and the work is a single pass, or when the operation is not relational — fitting a model, bootstrapping, plotting; <code>setkey()</code> gives it the same lookup advantage in memory. The two are not rivals: load with SQL, analyse in R, and use <code>dbplyr</code> when you want one pipeline to serve both.</p></details>

<details><summary>Q11. A website has the data you need and no API. List, in order, what you do.</summary>
<p>(1) Look for a bulk download or an official client — most large resources publish one. (2) Read the terms of use and <code>robots.txt</code>; check whether your paths are allowed and what crawl delay is requested. (3) Ask the data owner; a polite email often returns a file. (4) Only then consider a rate-limited, cached, identified crawl of the permitted paths — and never for personal data, paywalled content, or a service that would be degraded by it.</p></details>

<details><summary>Q12. The live record count for the target differs from the frozen snapshot's. Is something wrong?</summary>
<p>Not necessarily, and there are two innocent explanations before any guilty one. A curated, versioned database releases new content twice a year, so a later release legitimately returns more records; and the two queries may not be identical — the frozen snapshot of this course was built from a query over IC50, Ki and Kd, while the live call in Section 4 asks only for IC50. What would be wrong is reporting "data from ChEMBL" without the release, query and date, so that the difference could not be attributed to the database rather than to your pipeline. Record the release and the checksum (Unit 2) and the discrepancy becomes information instead of a mystery.</p></details>

## Glossary

<dl class="gloss">
<dt>Backoff (exponential)</dt><dd>Waiting a doubling interval between retries, with jitter, so that transient failures resolve without overloading a service.</dd>
<dt>CID / SID</dt><dd>PubChem's compound (standardised) and substance (as deposited) identifiers.</dd>
<dt>Columnar format</dt><dd>A layout storing each column contiguously (Parquet), so that reading a subset of columns reads a subset of the file.</dd>
<dt>DBI</dt><dd>R's common database interface; <code>RSQLite</code>, <code>RPostgres</code> and <code>duckdb</code> are drivers behind the same functions.</dd>
<dt>Fixed-column format</dt><dd>A text format in which each field occupies fixed character positions (PDB, SDF atom blocks); parsed with <code>substr()</code>, never by splitting on whitespace.</dd>
<dt>Foreign key</dt><dd>A column whose values refer to another table's primary key.</dd>
<dt>HDF5</dt><dd>A binary container for large, possibly chunked and compressed numeric arrays; read in R through <code>rhdf5</code>.</dd>
<dt>InChIKey</dt><dd>A 27-character hashed structural identifier: 14-character skeleton block, 10-character stereochemistry/isotope block, one protonation character.</dd>
<dt>Index (database)</dt><dd>A sorted auxiliary structure turning a table scan into a key lookup; costs storage and write speed. <code>data.table::setkey()</code> is the in-memory analogue.</dd>
<dt>JSON</dt><dd>A text format for nested data; the standard payload of REST APIs, read in R with <code>jsonlite</code>.</dd>
<dt>NULL (SQL)</dt><dd>SQL's marker for an absent value; comparisons with it are never true, exactly as with R's <code>NA</code>.</dd>
<dt>Pagination</dt><dd>Returning a large result set in successive pages, each with a pointer to the next.</dd>
<dt>Parquet</dt><dd>A typed, compressed, columnar file format for tables, read and written in R with <code>arrow</code>.</dd>
<dt>Primary key</dt><dd>The column (or combination) that uniquely identifies a row.</dd>
<dt>RDS</dt><dd>R's own serialisation of a single object; right for caching a fitted model, wrong for exchanging a table.</dd>
<dt>REST</dt><dd>An architectural style in which resources are addressed by URL and represented, usually, as JSON.</dd>
<dt>robots.txt</dt><dd>A server-published policy stating which paths automated clients may fetch (RFC 9309).</dd>
<dt>Row inflation</dt><dd>The multiplication of rows caused by joining on a key that is not unique on one side.</dd>
<dt>SDF</dt><dd>A multi-record chemical text format carrying atoms, bonds, coordinates and named data fields, terminated by <code>$$$$</code>.</dd>
<dt>SMILES</dt><dd>A one-line text encoding of a molecular graph.</dd>
<dt>Subquery</dt><dd>A query used as a table inside another query.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r2">Gaulton A, Bellis LJ, Bento AP, et al. ChEMBL: a large-scale bioactivity database for drug discovery. <em>Nucleic Acids Res</em>. 2012;40(D1):D1100-D1107. <a href="https://doi.org/10.1093/nar/gkr777">doi:10.1093/nar/gkr777</a></li>
<li id="r3">Bento AP, Gaulton A, Hersey A, et al. The ChEMBL bioactivity database: an update. <em>Nucleic Acids Res</em>. 2014;42(D1):D1083-D1090. <a href="https://doi.org/10.1093/nar/gkt1031">doi:10.1093/nar/gkt1031</a></li>
<li id="r4">Papadatos G, Gaulton A, Hersey A, Overington JP. Activity, assay and target data curation and quality in the ChEMBL database. <em>J Comput Aided Mol Des</em>. 2015;29(9):885-896. <a href="https://doi.org/10.1007/s10822-015-9860-5">doi:10.1007/s10822-015-9860-5</a></li>
<li id="r5">Kim S, Chen J, Cheng T, et al. PubChem 2023 update. <em>Nucleic Acids Res</em>. 2023;51(D1):D1373-D1380. <a href="https://doi.org/10.1093/nar/gkac956">doi:10.1093/nar/gkac956</a></li>
<li id="r6">Kim S, Thiessen PA, Bolton EE, et al. PubChem substance and compound databases. <em>Nucleic Acids Res</em>. 2016;44(D1):D1202-D1213. <a href="https://doi.org/10.1093/nar/gkv951">doi:10.1093/nar/gkv951</a></li>
<li id="r7">Gilson MK, Liu T, Baitaluk M, Nicola G, Hwang L, Chong J. BindingDB in 2015: a public database for medicinal chemistry, computational chemistry and systems pharmacology. <em>Nucleic Acids Res</em>. 2016;44(D1):D1045-D1053. <a href="https://doi.org/10.1093/nar/gkv1072">doi:10.1093/nar/gkv1072</a></li>
<li id="r8">UniProt Consortium. UniProt: the Universal Protein Knowledgebase in 2023. <em>Nucleic Acids Res</em>. 2023;51(D1):D523-D531. <a href="https://doi.org/10.1093/nar/gkac1052">doi:10.1093/nar/gkac1052</a></li>
<li id="r9">Burley SK, Bhikadiya C, Bi C, et al. RCSB Protein Data Bank: delivery of experimentally-determined PDB structures alongside one million computed structure models. <em>Nucleic Acids Res</em>. 2023;51(D1):D488-D508. <a href="https://doi.org/10.1093/nar/gkac1077">doi:10.1093/nar/gkac1077</a></li>
<li id="r10">Berman HM, Westbrook J, Feng Z, et al. The Protein Data Bank. <em>Nucleic Acids Res</em>. 2000;28(1):235-242. <a href="https://doi.org/10.1093/nar/28.1.235">doi:10.1093/nar/28.1.235</a></li>
<li id="r11">Barrett T, Wilhite SE, Ledoux P, et al. NCBI GEO: archive for functional genomics data sets — update. <em>Nucleic Acids Res</em>. 2013;41(D1):D991-D995. <a href="https://doi.org/10.1093/nar/gks1193">doi:10.1093/nar/gks1193</a></li>
<li id="r12">Edgar R, Domrachev M, Lash AE. Gene Expression Omnibus: NCBI gene expression and hybridization array data repository. <em>Nucleic Acids Res</em>. 2002;30(1):207-210. <a href="https://doi.org/10.1093/nar/30.1.207">doi:10.1093/nar/30.1.207</a></li>
<li id="r13">Sayers EW, Beck J, Bolton EE, et al. Database resources of the National Center for Biotechnology Information. <em>Nucleic Acids Res</em>. 2024;52(D1):D33-D43. <a href="https://doi.org/10.1093/nar/gkad1044">doi:10.1093/nar/gkad1044</a></li>
<li id="r14">Weininger D. SMILES, a chemical language and information system. 1. Introduction to methodology and encoding rules. <em>J Chem Inf Comput Sci</em>. 1988;28(1):31-36. <a href="https://doi.org/10.1021/ci00057a005">doi:10.1021/ci00057a005</a></li>
<li id="r15">Dalby A, Nourse JG, Hounshell WD, et al. Description of several chemical structure file formats used by computer programs developed at Molecular Design Limited. <em>J Chem Inf Comput Sci</em>. 1992;32(3):244-255. <a href="https://doi.org/10.1021/ci00007a012">doi:10.1021/ci00007a012</a></li>
<li id="r16">Pearson WR, Lipman DJ. Improved tools for biological sequence comparison. <em>Proc Natl Acad Sci USA</em>. 1988;85(8):2444-2448. <a href="https://doi.org/10.1073/pnas.85.8.2444">doi:10.1073/pnas.85.8.2444</a></li>
<li id="r17">Bourne PE, Berman HM, McMahon B, Watenpaugh KD, Westbrook JD, Fitzgerald PMD. Macromolecular crystallographic information file. <em>Methods Enzymol</em>. 1997;277:571-590. <a href="https://doi.org/10.1016/S0076-6879(97)77032-0">doi:10.1016/S0076-6879(97)77032-0</a></li>
<li id="r18">Folk M, Heber G, Koziol Q, Pourmal E, Robinson D. An overview of the HDF5 technology suite and its applications. <em>Proc EDBT/ICDT Workshop on Array Databases</em>. 2011:36-47. <a href="https://doi.org/10.1145/1966895.1966900">doi:10.1145/1966895.1966900</a></li>
<li id="r19">Vohra D. Apache Parquet. In: <em>Practical Hadoop Ecosystem</em>. Apress; 2016:325-335. <a href="https://doi.org/10.1007/978-1-4842-2199-0_8">doi:10.1007/978-1-4842-2199-0_8</a></li>
<li id="r20">Bray T, ed. The JavaScript Object Notation (JSON) data interchange format. <em>RFC 7159, IETF</em>. 2014. <a href="https://doi.org/10.17487/RFC7159">doi:10.17487/RFC7159</a></li>
<li id="r21">Fielding RT, Taylor RN. Principled design of the modern Web architecture. <em>ACM Trans Internet Technol</em>. 2002;2(2):115-150. <a href="https://doi.org/10.1145/514183.514185">doi:10.1145/514183.514185</a></li>
<li id="r22">Heller SR, McNaught A, Pletnev I, Stein S, Tchekhovskoi D. InChI, the IUPAC International Chemical Identifier. <em>J Cheminform</em>. 2015;7:23. <a href="https://doi.org/10.1186/s13321-015-0068-4">doi:10.1186/s13321-015-0068-4</a></li>
<li id="r23">Chambers J, Davies M, Gaulton A, et al. UniChem: a unified chemical structure cross-referencing and identifier tracking system. <em>J Cheminform</em>. 2013;5:3. <a href="https://doi.org/10.1186/1758-2946-5-3">doi:10.1186/1758-2946-5-3</a></li>
<li id="r24">Codd EF. A relational model of data for large shared data banks. <em>Commun ACM</em>. 1970;13(6):377-387. <a href="https://doi.org/10.1145/362384.362685">doi:10.1145/362384.362685</a></li>
<li id="r25">Gaffney KP, Prammer M, Brasfield L, Hipp DR, Kennedy D, Patel JM. SQLite: past, present, and future. <em>Proc VLDB Endow</em>. 2022;15(12):3535-3547. <a href="https://doi.org/10.14778/3554821.3554842">doi:10.14778/3554821.3554842</a></li>
<li id="r26">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public K<sub>i</sub> data. <em>J Med Chem</em>. 2012;55(11):5165-5173. <a href="https://doi.org/10.1021/jm300131x">doi:10.1021/jm300131x</a></li>
<li id="r27">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data — a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r28">Tiikkainen P, Bellis L, Light Y, Franke L. Estimating error rates in bioactivity databases. <em>J Chem Inf Model</em>. 2013;53(10):2499-2505. <a href="https://doi.org/10.1021/ci400099q">doi:10.1021/ci400099q</a></li>
<li id="r29">Koster M, Illyes G, Zeller H, Sassman L. Robots exclusion protocol. <em>RFC 9309, IETF</em>. 2022. <a href="https://doi.org/10.17487/RFC9309">doi:10.17487/RFC9309</a></li>
<li id="r30">Wickham H. Tidy data. <em>J Stat Softw</em>. 2014;59(10):1-23. <a href="https://doi.org/10.18637/jss.v059.i10">doi:10.18637/jss.v059.i10</a></li>
<li id="r31">Williams AJ, Ekins S. A quality alert and call for improved curation of public chemistry databases. <em>Drug Discov Today</em>. 2011;16(17-18):747-750. <a href="https://doi.org/10.1016/j.drudis.2011.07.007">doi:10.1016/j.drudis.2011.07.007</a></li>
<li id="r32">Davies M, Nowotka M, Papadatos G, et al. ChEMBL web services: streamlining access to drug discovery data and utilities. <em>Nucleic Acids Res</em>. 2015;43(W1):W612-W620. <a href="https://doi.org/10.1093/nar/gkv352">doi:10.1093/nar/gkv352</a></li>
<li id="r33">Mendez D, Gaulton A, Bento AP, et al. ChEMBL: towards direct deposition of bioassay data. <em>Nucleic Acids Res</em>. 2019;47(D1):D930-D940. <a href="https://doi.org/10.1093/nar/gky1075">doi:10.1093/nar/gky1075</a></li>
<li id="r34">Wickham H, Averick M, Bryan J, et al. Welcome to the tidyverse. <em>J Open Source Softw</em>. 2019;4(43):1686. <a href="https://doi.org/10.21105/joss.01686">doi:10.21105/joss.01686</a></li>
<li id="r35">Charif D, Lobry JR. SeqinR 1.0-2: a contributed package to the R project for statistical computing devoted to biological sequences retrieval and analysis. In: <em>Structural Approaches to Sequence Evolution</em>. Springer; 2007:207-232. <a href="https://doi.org/10.1007/978-3-540-35306-5_10">doi:10.1007/978-3-540-35306-5_10</a></li>
<li id="r36">Cao Y, Charisi A, Cheng LC, Jiang T, Girke T. ChemmineR: a compound mining framework for R. <em>Bioinformatics</em>. 2008;24(15):1733-1734. <a href="https://doi.org/10.1093/bioinformatics/btn307">doi:10.1093/bioinformatics/btn307</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023. (Free online; the "Import" chapters are the companion reading for Sections 2 and 3.)</li>
<li>Beaulieu A. <em>Learning SQL</em>. 3rd ed. O'Reilly; 2020. (Chapters 3–8 cover everything in Section 6 at greater length.)</li>
<li>Wickham H. <em>Advanced R</em>. 2nd ed. Chapman and Hall/CRC; 2019. (Free online; the chapter on conditions is the reference for the error handling of Section 4.)</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><a href="https://www.ebi.ac.uk/chembl/api/data/docs">ChEMBL web-services documentation</a> — the API called in Section 4.</li>
<li><a href="https://chembl.gitbook.io/chembl-interface-documentation/downloads">ChEMBL downloads</a> — the SQLite and PostgreSQL dumps to use instead of paging an API at scale.</li>
<li><a href="https://httr2.r-lib.org/">httr2</a>, <a href="https://arrow.apache.org/docs/r/">arrow for R</a>, <a href="https://dbi.r-dbi.org/">DBI</a> and <a href="https://dbplyr.tidyverse.org/">dbplyr</a> — the four package references for this unit.</li>
<li><a href="https://www.sqlite.org/docs.html">SQLite documentation</a> and the <a href="https://rdatatable.gitlab.io/data.table/">data.table vignettes</a> on keys and indices.</li>
<li><a href="https://datacarpentry.org/lessons/">Data Carpentry</a> — the "SQL for ecology/genomics" lessons, which are the same SQL on different data.</li>
</ul>
